In [18]:
!pip install -q \
utilsforecast \
statsforecast \
holidays \
hyperopt \
category_encoders \
sktime \
shap \
lightgbm \
catboost \
xgboost \
plotly \
tqdm

## Конфигурация

In [19]:
from pathlib import Path
import os

# ============================================================
# Environment
# ============================================================

IN_KAGGLE = False
IN_COLAB = False
LOCAL_RUN = True

# Текущая рабочая директория, откуда запущен ноутбук / скрипт
PROJECT_ROOT = Path.cwd()

# ============================================================
# Global config
# ============================================================

RANDOM_STATE = 42

VALID_YEAR = 2025
VALID_MONTH = 2

FORECAST_YEAR = 2025
FORECAST_MONTH = 3

EXPECTED_TEST_SIZE = 18_657

# ============================================================
# CatBoost config
# ============================================================

# Если локально есть настроенный GPU для CatBoost, оставляем GPU.
# Если будет ошибка с CUDA/GPU, поменяй на "CPU".
CATBOOST_TASK_TYPE = "GPU"
CATBOOST_DEVICES = None

RUN_PERMUTATION_IMPORTANCE = False

# ============================================================
# Local paths
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "90_13_outputs"

SUBMISSION_PATH = OUTPUT_DIR / "test.csv"
SHAP_IMPORTANCE_PATH = OUTPUT_DIR / "shap_importance.csv"
MODEL_DIR = OUTPUT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("LOCAL_RUN:", LOCAL_RUN)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SUBMISSION_PATH:", SUBMISSION_PATH)
print("SHAP_IMPORTANCE_PATH:", SHAP_IMPORTANCE_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("CATBOOST_TASK_TYPE:", CATBOOST_TASK_TYPE)
print("CATBOOST_DEVICES:", CATBOOST_DEVICES)

LOCAL_RUN: True
PROJECT_ROOT: /kaggle/working
OUTPUT_DIR: /kaggle/working/90_13_outputs
SUBMISSION_PATH: /kaggle/working/90_13_outputs/test.csv
SHAP_IMPORTANCE_PATH: /kaggle/working/90_13_outputs/shap_importance.csv
MODEL_DIR: /kaggle/working/90_13_outputs/models
CATBOOST_TASK_TYPE: GPU
CATBOOST_DEVICES: None


## Импорты

In [20]:
# Базовые библиотеки и утилиты
import numpy as np
import pandas as pd
import datetime
import os
import time
import time as time_module
import warnings
from copy import deepcopy
from bisect import bisect_left
from itertools import product
from typing import Union, Optional

# Настройки pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_columns', None)

# Отображение в Jupyter
from IPython.display import display
%matplotlib inline

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from utilsforecast.plotting import plot_series

# Работа с календарём
import holidays

# Прогресс-бары
from tqdm import tqdm

# SciPy
import scipy as sp
from scipy import stats
from scipy.stats import ttest_1samp, shapiro, ks_2samp, zscore
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.datasets import longley
from statsmodels.formula.api import ols
from statsmodels.graphics import tsaplots
from statsmodels.graphics.gofplots import qqplot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ar_model import ar_select_order
from statsmodels.tsa.arima.model import ARIMA

# StatsForecast
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, _TS

# Scikit-learn: модели, препроцессинг, метрики
from sklearn.cluster import DBSCAN
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, mean_absolute_error, mean_squared_error, precision_score,
    r2_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    GridSearchCV, ParameterGrid, RandomizedSearchCV,
    TimeSeriesSplit, train_test_split
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Boosting модели
from catboost import CatBoostRegressor
import catboost as cb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRegressor
import xgboost as xgb

# Гиперпараметрический поиск
from hyperopt import fmin, tpe, Trials, STATUS_OK, hp

# Кодировщики категориальных признаков
import category_encoders as ce

# # Дополнительные утилиты
# try:
#     from sktime.utils.plotting import plot_correlations
# except ModuleNotFoundError:
#     plot_correlations = None
#     print("Optional dependency sktime is not installed; plot_correlations is disabled.")

# SHAP
import shap

## Чтение данных

In [21]:
# df = pd.read_csv(SOURCE_DATA_PATH)
# print("Raw data shape:", df.shape)
# df.head()

In [22]:
df = pd.read_csv('/kaggle/input/datasets/tedacrawford/train-2/train_2.csv')
df.head()

,new_id,Год,Месяц,Среднее количество промо товаров в чеке,Среднее количество товаров в чеке,Среднее количество отмен,Рабочие часы в день,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,2024,1,1.080,6.030,147.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,75147744.850
1,0,2023,1,1.320,6.040,162.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74914754.220
2,0,2025,1,0.820,6.000,145.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,87125506.920
3,0,2025,2,0.900,6.000,118.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,82659801.630
4,0,2024,2,1.250,6.060,154.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74209339.110


In [23]:
rename_cols = {
    "new_id": "new_id",
    "Год": "year",
    "Месяц": "month",
    "Среднее количество промо товаров в чеке": "avg_promo_items_per_receipt",
    "Среднее количество товаров в чеке": "avg_items_per_receipt",
    "Среднее количество отмен": "avg_cancellations",
    "Рабочие часы в день": "working_hours_per_day",
    "Дата открытия, категориальный": "opening_date_category",
    "Торговая площадь, категориальный": "trading_area_category",
    "Населенный пункт": "settlement",
    "Регион": "region",
    "Численность населения": "population_size",
    "Количество домохозяйств": "number_of_households",
    "Трафик пеший, в час": "pedestrian_traffic_per_hour",
    "Трафик авто, в час": "car_traffic_per_hour",
    "Маркетплейсы, доставки, постаматы (100 м)": "marketplaces_deliveries_parcel_lockers_100m",
    "Медицинские уч. и аптеки (300 м)": "medical_institutions_and_pharmacies_300m",
    "Школы (300 м)": "schools_300m",
    "Остановки (300 м)": "public_transport_stops_300m",
    "Продуктовые магазины (500 м)": "grocery_stores_500m",
    "Пятерочки (500 м)": "pyaterochka_stores_500m",
    "Количество касс": "number_of_cash_registers",
    "Флаг алкогольной лицензии": "alcohol_license_flag",
    "РТО": "pto",
}

df = df.rename(columns=rename_cols)

print("Renamed data shape:", df.shape)
df.head()

Renamed data shape: (485082, 24)


,new_id,year,month,avg_promo_items_per_receipt,avg_items_per_receipt,avg_cancellations,working_hours_per_day,opening_date_category,trading_area_category,settlement,region,population_size,number_of_households,pedestrian_traffic_per_hour,car_traffic_per_hour,marketplaces_deliveries_parcel_lockers_100m,medical_institutions_and_pharmacies_300m,schools_300m,public_transport_stops_300m,grocery_stores_500m,pyaterochka_stores_500m,number_of_cash_registers,alcohol_license_flag,pto
0,0,2024,1,1.080,6.030,147.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,75147744.850
1,0,2023,1,1.320,6.040,162.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74914754.220
2,0,2025,1,0.820,6.000,145.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,87125506.920
3,0,2025,2,0.900,6.000,118.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,82659801.630
4,0,2024,2,1.250,6.060,154.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74209339.110


## Функции создания признаков

In [24]:
from typing import Literal, Optional, Sequence

import gc
import warnings
import numpy as np
import pandas as pd


def make_panel_time_split(
    df: pd.DataFrame,
    *,
    group_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    min_year: Optional[int] = None,
    create_time_col: bool = True,
    create_test_if_missing: bool = True,
    test_target_value=np.nan,
    future_unknown_cols: Optional[Sequence[str]] = None,
    drop_target_na_from_train: bool = True,
    require_known_val_target: bool = True,
    check_duplicates: bool = True,
    validate_time_col: bool = True,
    expected_test_size: Optional[int] = None,
    copy: bool = True,
    verbose: bool = True,
):
    """
    Leakage-safe time split для panel monthly forecasting.

    split_mode:
    - "train_val": train = months before val, val = selected validation month;
    - "train_val_test": train + val + artificial/existing test month;
    - "train_test": train = all known history before test, test = artificial/existing test month.

    Если test-месяца нет в df, он создаётся из последней известной строки каждого магазина.
    Важно: если дальше строятся лаги/rolling, test-строки должны быть созданы ДО feature engineering.
    """
    if split_mode not in {"train_val", "train_val_test", "train_test"}:
        raise ValueError("split_mode должен быть 'train_val', 'train_val_test' или 'train_test'.")

    if copy:
        df = df.copy()

    def _check_time_args(prefix, time_idx, year, month):
        if time_idx is not None and (year is not None or month is not None):
            raise ValueError(
                f"Для {prefix} передай либо {prefix}_time_idx, либо пару "
                f"{prefix}_year + {prefix}_month, но не оба варианта."
            )
        if (year is None) ^ (month is None):
            raise ValueError(
                f"Для {prefix} нужно передавать year и month вместе. "
                f"Получено: {prefix}_year={year}, {prefix}_month={month}."
            )

    _check_time_args("val", val_time_idx, val_year, val_month)
    _check_time_args("test", test_time_idx, test_year, test_month)

    if split_mode == "train_test" and (
        val_time_idx is not None or val_year is not None or val_month is not None
    ):
        raise ValueError("В split_mode='train_test' validation-месяц не используется.")

    required_cols = {group_col, target_col}
    has_time_col = time_col in df.columns
    has_year_month = year_col in df.columns and month_col in df.columns

    if not has_time_col:
        if create_time_col:
            required_cols.update({year_col, month_col})
        else:
            required_cols.add(time_col)

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"В df отсутствуют обязательные колонки: {sorted(missing)}")

    if has_year_month:
        if df[year_col].isna().any() or df[month_col].isna().any():
            raise ValueError(f"В {year_col}/{month_col} есть пропуски.")
        df[year_col] = pd.to_numeric(df[year_col], errors="raise").astype(int)
        df[month_col] = pd.to_numeric(df[month_col], errors="raise").astype(int)
        bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
        if bad_months:
            raise ValueError(f"В колонке '{month_col}' некорректные месяцы: {bad_months}.")

    if has_time_col:
        if df[time_col].isna().any():
            raise ValueError(f"В колонке '{time_col}' есть пропуски.")
        df[time_col] = pd.to_numeric(df[time_col], errors="raise").astype(int)

        if min_year is None and has_year_month:
            inferred = df[year_col] - ((df[time_col] - (df[month_col] - 1)) // 12)
            unique = sorted(inferred.unique())
            if len(unique) == 1:
                min_year = int(unique[0])
            else:
                raise ValueError(
                    f"Не удалось однозначно восстановить min_year из {time_col}. "
                    f"Передай min_year явно. Найдено: {unique[:10]}"
                )

        if validate_time_col and has_year_month and min_year is not None:
            expected = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)
            bad = df[time_col].astype(int) != expected
            if bad.any():
                examples = df.loc[bad, [group_col, year_col, month_col, time_col]].head(10)
                raise ValueError(
                    f"'{time_col}' не совпадает с расчётом по min_year={min_year}. "
                    f"Примеры:\n{examples}"
                )
    else:
        if not has_year_month:
            raise ValueError(f"Нельзя создать '{time_col}', потому что нет year/month.")
        if min_year is None:
            min_year = int(df[year_col].min())
        df[time_col] = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)

    known_target_times = sorted(df.loc[df[target_col].notna(), time_col].unique())
    if not known_target_times:
        raise ValueError(f"В '{target_col}' нет известных значений.")
    max_known_time_idx = int(known_target_times[-1])

    def _resolve(label, time_idx, year, month, default=None):
        if time_idx is not None:
            return int(time_idx)
        if year is not None and month is not None:
            if min_year is None:
                raise ValueError(f"Для расчёта {label}_time_idx нужен min_year.")
            month = int(month)
            if not 1 <= month <= 12:
                raise ValueError(f"{label}_month должен быть от 1 до 12.")
            return (int(year) - int(min_year)) * 12 + (month - 1)
        return None if default is None else int(default)

    if split_mode in {"train_val", "train_val_test"}:
        val_time_idx = _resolve("val", val_time_idx, val_year, val_month, max_known_time_idx)
    else:
        val_time_idx = None

    if split_mode in {"train_val_test", "train_test"}:
        test_time_idx = _resolve("test", test_time_idx, test_year, test_month, max_known_time_idx + 1)
    else:
        test_time_idx = None

    if split_mode == "train_val_test" and not (val_time_idx < test_time_idx):
        raise ValueError(
            f"Ожидается val_time_idx < test_time_idx, получено "
            f"{val_time_idx=} и {test_time_idx=}."
        )

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)

    if check_duplicates:
        dup_mask = df.duplicated([group_col, time_col], keep=False)
        if dup_mask.any():
            examples = df.loc[dup_mask, [group_col, time_col]].sort_values([group_col, time_col]).head(10)
            raise ValueError(
                f"Есть дубли по [{group_col}, {time_col}]. "
                f"Строк-дублей: {int(dup_mask.sum())}. Примеры:\n{examples}"
            )

    test = None
    if split_mode in {"train_val_test", "train_test"}:
        existing = df[df[time_col] == test_time_idx].copy()
        if not existing.empty:
            test = existing.copy()
        else:
            if not create_test_if_missing:
                raise ValueError(f"В df нет test_time_idx={test_time_idx}.")
            hist = df[(df[time_col] < test_time_idx) & (df[target_col].notna())].copy()
            if hist.empty:
                raise ValueError(f"Нет исторических строк до test_time_idx={test_time_idx}.")
            test = (
                hist.sort_values([group_col, time_col])
                .groupby(group_col, sort=False)
                .tail(1)
                .copy()
            )
            test[time_col] = int(test_time_idx)
            if has_year_month and min_year is not None:
                test[year_col] = int(min_year) + int(test_time_idx) // 12
                test[month_col] = int(test_time_idx) % 12 + 1
            test[target_col] = test_target_value

            if future_unknown_cols is not None:
                for col in future_unknown_cols:
                    if col in test.columns:
                        test[col] = np.nan
                    else:
                        warnings.warn(f"Колонка '{col}' из future_unknown_cols отсутствует в df.")

        test = test.sort_values([group_col, time_col]).reset_index(drop=True)
        if expected_test_size is not None and len(test) != int(expected_test_size):
            raise ValueError(
                f"Размер test не совпадает с expected_test_size: "
                f"получено {len(test)}, ожидалось {expected_test_size}."
            )

    if split_mode == "train_test":
        train_mask = df[time_col] < test_time_idx
    else:
        train_mask = df[time_col] < val_time_idx

    if drop_target_na_from_train:
        train_mask = train_mask & df[target_col].notna()

    train = df.loc[train_mask].copy()
    if train.empty:
        raise ValueError("train получился пустым.")

    val = None
    if split_mode in {"train_val", "train_val_test"}:
        val = df[df[time_col] == val_time_idx].copy()
        if val.empty:
            available = sorted(df[time_col].unique())
            raise ValueError(
                f"val пустой для val_time_idx={val_time_idx}. "
                f"Доступные time_idx: {available[:5]} ... {available[-5:]}"
            )
        if require_known_val_target and val[target_col].isna().any():
            n_nan = int(val[target_col].isna().sum())
            raise ValueError(
                f"В validation есть NaN в '{target_col}': {n_nan}. "
                "Скорее всего, validation попал на искусственный test-месяц."
            )
        val = val.sort_values([group_col, time_col]).reset_index(drop=True)

    train = train.sort_values([group_col, time_col]).reset_index(drop=True)

    if verbose:
        print("split_mode:", split_mode)
        print("min_year:", min_year)
        print("max_known_time_idx:", max_known_time_idx)
        if val_time_idx is not None:
            print("val_time_idx:", val_time_idx)
        if test_time_idx is not None:
            print("test_time_idx:", test_time_idx)
        print("train shape:", train.shape)
        if val is not None:
            print("val shape:", val.shape)
        if test is not None:
            print("test shape:", test.shape)

    if split_mode == "train_val":
        return train, val
    if split_mode == "train_test":
        return train, test
    return train, val, test




# ============================================================
# Base feature generator from uploaded file
# ============================================================

import gc
import numpy as np
import pandas as pd


def generate_retail_forecast_features_leakfree(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame | None = None,
    test_df: pd.DataFrame | None = None,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    make_val_from_train: bool = False,
    val_time_idx: int | None = None,
    use_val_as_test_history: bool = True,
    add_calendar_features: bool = True,
    add_static_features: bool = True,
    add_target_history: bool = True,
    add_category_target_history: bool = True,
    add_non_target_monthly_history: bool = True,
    include_current_receipt_features: bool = False,
    current_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    current_monthly_derived_cols: tuple[str, ...] = (
        "promo_items_share",
        "promo_x_items",
        "items_x_working_hours",
        "items_x_working_days",
        "alcohol_x_items",
    ),
    working_hours_future_known: bool = False,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    target_rolling_windows: tuple[int, ...] = (3, 12),
    target_rolling_min_periods: int = 2,
    target_history_scale: str = "raw",   # raw | log | both
    keep_raw_pto_lag_1_anchor_for_log: bool = True,
    add_log_target_dynamics: bool = True,
    group_specs: list | None = None,
    group_agg_stat: str = "mean",
    group_lag: int = 1,
    add_group_lag_feature: bool = True,
    add_group_relative_ratio: bool = True,
    add_group_relative_diff: bool = False,
    monthly_feature_config: dict | None = None,
    monthly_history_scale: str = "raw",  # raw | log | both
    monthly_rolling_min_periods: int = 2,
    add_monthly_cross_lags: bool = True,
    monthly_cross_lags: tuple[int, ...] = (1, 12),
    monthly_cross_feature_types: tuple[str, ...] = ("promo_share",),
    drop_current_monthly_cols: bool = True,
    drop_current_monthly_derived_cols: bool = True,
    drop_year_col: bool = True,
    drop_obvious_duplicates: bool = True,
    extra_drop_cols: list | tuple | None = None,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
    copy: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Единый leak-free feature-pipeline для panel monthly forecasting.

    Основной контракт:
    - val/test target не используется для лагов и group target history;
    - текущие monthly-cols из val/test не используются для лагов;
    - лаги считаются через time-aware merge по time_col - lag, а не через shift строк;
    - rolling-признаки считаются только по прошлым наблюдениям внутри history;
    - для test можно использовать train+val как историю, если val уже является известным прошлым.
    """

    if target_history_scale not in {"raw", "log", "both"}:
        raise ValueError("target_history_scale должен быть 'raw', 'log' или 'both'.")
    if monthly_history_scale not in {"raw", "log", "both"}:
        raise ValueError("monthly_history_scale должен быть 'raw', 'log' или 'both'.")
    if group_agg_stat not in {"mean", "median"}:
        raise ValueError("group_agg_stat должен быть 'mean' или 'median'.")

    if copy:
        train_df = train_df.copy()
        val_df = val_df.copy() if val_df is not None else None
        test_df = test_df.copy() if test_df is not None else None

    # ============================================================
    # Helpers
    # ============================================================

    def _safe_divide(numerator, denominator, fill=np.nan):
        numerator = numerator if isinstance(numerator, pd.Series) else pd.Series(numerator)
        denominator = denominator if isinstance(denominator, pd.Series) else pd.Series(denominator)
        result = numerator / denominator.replace(0, np.nan)
        result = result.replace([np.inf, -np.inf], np.nan)
        if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
            result = result.fillna(fill)
        return result

    def _col(df, name, default=0.0):
        if name in df.columns:
            return df[name].fillna(default)
        return pd.Series(default, index=df.index)

    def _ensure_time_year_month(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
        df = df.copy()

        has_year_month = year_col in df.columns and month_col in df.columns
        has_time = time_col in df.columns

        if not has_year_month and not has_time:
            raise ValueError(
                f"В {name} нет ни пары '{year_col}'+'{month_col}', ни '{time_col}'."
            )

        if has_year_month:
            if df[year_col].isna().any() or df[month_col].isna().any():
                raise ValueError(f"В {name} есть NA в {year_col}/{month_col}.")
            df[year_col] = df[year_col].astype(int)
            df[month_col] = df[month_col].astype(int)
            bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
            if bad_months:
                raise ValueError(f"В {name} некорректные месяцы: {bad_months}.")
            # Важно: всегда пересчитываем глобальную шкалу, чтобы не было local-min-year.
            df[time_col] = ((df[year_col] - min_year) * 12 + (df[month_col] - 1)).astype(int)
        else:
            if df[time_col].isna().any():
                raise ValueError(f"В {name} есть NA в {time_col}.")
            df[time_col] = df[time_col].astype(int)
            df[year_col] = (min_year + df[time_col] // 12).astype(int)
            df[month_col] = (df[time_col] % 12 + 1).astype(int)

        return df

    def _validate_required(df: pd.DataFrame, required: set[str], name: str):
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"В {name} нет обязательных колонок: {missing}")

    def _validate_unique_panel(df: pd.DataFrame, name: str):
        if {id_col, time_col}.issubset(df.columns):
            duplicated = df.duplicated([id_col, time_col]).sum()
            if duplicated > 0:
                raise ValueError(
                    f"В {name} найдено {duplicated} дублей по [{id_col}, {time_col}]. "
                    "Для лагов ожидается одна строка на магазин-месяц."
                )

    def _part_concat(parts: dict[str, pd.DataFrame]) -> pd.DataFrame:
        out = []
        for part_name, df_part in parts.items():
            p = df_part.copy()
            p["__part__"] = part_name
            p["__row_order__"] = np.arange(len(p))
            out.append(p)
        return pd.concat(out, axis=0, ignore_index=True, sort=False)

    def _split_parts(combined: pd.DataFrame, part_names: list[str]) -> dict[str, pd.DataFrame]:
        result = {}
        for part_name in part_names:
            part = (
                combined[combined["__part__"] == part_name]
                .sort_values("__row_order__")
                .drop(columns=["__part__", "__row_order__"], errors="ignore")
                .reset_index(drop=True)
            )
            result[part_name] = part
        return result

    def _drop_preexisting_generated_cols(df: pd.DataFrame) -> pd.DataFrame:
        prefixes = [
            f"{target_col}_lag_", f"log_{target_col}_lag_", f"{target_col}_roll_",
            f"{target_col}_diff_", f"{target_col}_ratio_", f"{target_col}_seasonal_",
            f"log_{target_col}_diff_", f"{target_col}_lag_1_to_", f"{target_col}_lag_12_to_",
            "promo_items_share_lag_", "promo_x_items_lag_", "items_x_working_hours_lag_",
            "cancellations_per_item_lag_", "cancellations_per_working_hour_lag_",
            "region_", "settlement_",
        ]
        monthly_prefixes = []
        for c in current_monthly_cols:
            monthly_prefixes.extend([
                f"{c}_lag_", f"log_{c}_lag_", f"{c}_roll_", f"{c}_diff_", f"{c}_ratio_",
            ])

        explicit = {
            "season", "is_winter", "is_spring", "is_autumn", "is_quarter_start_month",
            "is_quarter_end_month", "is_halfyear_start_month", "is_halfyear_end_month",
            "is_year_start_month", "is_year_end_month", "month_in_halfyear",
            "month_name", "month_sin", "month_cos", "month_sin_2", "month_cos_2",
            "prev_month", "n_days_in_month", "n_weekend_days", "n_may_holiday_days",
            "n_single_public_holiday_days", "n_november_holiday_days",
            "n_december_additional_holiday_days", "n_holiday_days", "n_additional_holiday_days",
            "n_working_days", "n_additional_working_days", "is_leap_year", "holiday_month_type",
            "is_middle_of_quarter", "is_pre_new_year_month", "is_pre_may_holidays_month",
            "is_back_to_school_month", "is_business_activity_low_month", "is_business_activity_high_month",
            "total_traffic_per_hour", "pedestrian_traffic_share", "car_to_pedestrian_traffic_ratio",
            "log_population_size", "log_number_of_households", "log_pedestrian_traffic_per_hour",
            "log_car_traffic_per_hour", "is_moscow", "is_million_plus_city", "population_city_type",
            "has_school_nearby", "has_transport_nearby", "has_marketplace_nearby",
            "has_medical_nearby", "has_grocery_competition", "has_pyaterochka_nearby",
            "is_high_grocery_competition", "is_high_pyaterochka_density",
            "non_pyaterochka_grocery_stores_500m", "pyaterochka_share_among_grocery",
            "non_pyaterochka_competition_share", "grocery_per_1000_people",
            "pyaterochka_per_1000_people", "competition_per_1000_households",
            "pyaterochka_per_1000_households", "people_per_household",
            "households_per_1000_people", "traffic_per_1000_people", "traffic_per_1000_households",
            "pedestrian_traffic_per_1000_people", "social_infra_score", "commercial_infra_score",
            "poi_score", "poi_per_1000_households", "public_transport_share_in_social_infra",
            "medical_share_in_social_infra", "cash_register_hours_capacity", "traffic_per_cash_register",
            "traffic_per_working_hour", "households_per_cash_register", "population_per_cash_register",
            "cash_registers_per_total_traffic", "promo_items_share", "promo_x_items",
            "items_x_working_hours", "items_x_working_days", "alcohol_x_capacity",
            "alcohol_x_total_traffic", "alcohol_x_items", "traffic_x_total_poi",
            "traffic_x_grocery_competition", "traffic_x_pyaterochka_competition",
            "pedestrian_traffic_x_transport", "schools_x_back_to_school",
            "settlement_store_count", "region_store_count", "region_trading_area_store_count",
            "region_opening_age_store_count",
        }

        drop_cols = []
        for c in df.columns:
            if c in explicit:
                drop_cols.append(c)
            elif any(c.startswith(p) for p in prefixes + monthly_prefixes):
                # Не удаляем сырые region/settlement, только generated mean/count; это грубый фильтр ниже уточняем.
                if c in {"region", "settlement"}:
                    continue
                # region_* может быть исходной колонкой только маловероятно; для безопасности удаляем только *_mean/count.
                if c.startswith(("region_", "settlement_")) and not (c.endswith("_mean") or c.endswith("_count")):
                    continue
                drop_cols.append(c)
            elif f"_{target_col}_{group_agg_stat}_lag_" in c:
                drop_cols.append(c)

        return df.drop(columns=sorted(set(drop_cols)), errors="ignore")

    # ============================================================
    # Calendar and static feature blocks
    # ============================================================

    def _add_calendar(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        years = sorted(df[year_col].dropna().astype(int).unique())
        if not years:
            return df

        def _holiday_calendar():
            holidays = {
                # 2023
                "2023-01-01": "New_Year_Holidays", "2023-01-02": "New_Year_Holidays",
                "2023-01-03": "New_Year_Holidays", "2023-01-04": "New_Year_Holidays",
                "2023-01-05": "New_Year_Holidays", "2023-01-06": "New_Year_Holidays",
                "2023-01-07": "Christmas_Day", "2023-01-08": "New_Year_Holidays",
                "2023-02-23": "Defender_of_the_Fatherland_Day",
                "2023-03-08": "International_Womens_Day",
                "2023-05-01": "Spring_and_Labour_Holiday", "2023-05-09": "Victory_Day",
                "2023-06-12": "Day_of_Russia", "2023-11-04": "National_Unity_Day",
                "2023-02-24": "Additional_Holiday", "2023-05-08": "Additional_Holiday",
                "2023-11-06": "Additional_Holiday",
                # 2024
                "2024-01-01": "New_Year_Holidays", "2024-01-02": "New_Year_Holidays",
                "2024-01-03": "New_Year_Holidays", "2024-01-04": "New_Year_Holidays",
                "2024-01-05": "New_Year_Holidays", "2024-01-06": "New_Year_Holidays",
                "2024-01-07": "Christmas_Day", "2024-01-08": "New_Year_Holidays",
                "2024-02-23": "Defender_of_the_Fatherland_Day",
                "2024-03-08": "International_Womens_Day",
                "2024-05-01": "Spring_and_Labour_Holiday", "2024-05-09": "Victory_Day",
                "2024-06-12": "Day_of_Russia", "2024-11-04": "National_Unity_Day",
                "2024-04-29": "Additional_Holiday", "2024-04-30": "Additional_Holiday",
                "2024-05-10": "Additional_Holiday", "2024-12-30": "Additional_Holiday",
                "2024-12-31": "Additional_Holiday",
                # 2025
                "2025-01-01": "New_Year_Holidays", "2025-01-02": "New_Year_Holidays",
                "2025-01-03": "New_Year_Holidays", "2025-01-04": "New_Year_Holidays",
                "2025-01-05": "New_Year_Holidays", "2025-01-06": "New_Year_Holidays",
                "2025-01-07": "Christmas_Day", "2025-01-08": "New_Year_Holidays",
                "2025-02-23": "Defender_of_the_Fatherland_Day",
                "2025-03-08": "International_Womens_Day",
                "2025-05-01": "Spring_and_Labour_Holiday", "2025-05-09": "Victory_Day",
                "2025-06-12": "Day_of_Russia", "2025-11-04": "National_Unity_Day",
                "2025-05-02": "Additional_Holiday", "2025-05-08": "Additional_Holiday",
                "2025-06-13": "Additional_Holiday", "2025-11-03": "Additional_Holiday",
                "2025-12-31": "Additional_Holiday",
            }
            additional_working_days = {"2024-04-27", "2024-11-02", "2024-12-28", "2025-11-01"}
            return (
                {pd.to_datetime(k): v for k, v in holidays.items()},
                {pd.to_datetime(x) for x in additional_working_days},
            )

        holidays, additional_working_days = _holiday_calendar()

        df["season"] = df[month_col].map({
            12: "Winter", 1: "Winter", 2: "Winter",
            3: "Spring", 4: "Spring", 5: "Spring",
            6: "Summer", 7: "Summer", 8: "Summer",
            9: "Autumn", 10: "Autumn", 11: "Autumn",
        })
        df["is_winter"] = (df["season"] == "Winter").astype(int)
        df["is_spring"] = (df["season"] == "Spring").astype(int)
        df["is_autumn"] = (df["season"] == "Autumn").astype(int)
        df["is_quarter_start_month"] = df[month_col].isin([1, 4, 7, 10]).astype(int)
        df["is_quarter_end_month"] = df[month_col].isin([3, 6, 9, 12]).astype(int)
        df["is_halfyear_start_month"] = df[month_col].isin([1, 7]).astype(int)
        df["is_halfyear_end_month"] = df[month_col].isin([6, 12]).astype(int)
        df["is_year_start_month"] = (df[month_col] == 1).astype(int)
        df["is_year_end_month"] = (df[month_col] == 12).astype(int)
        df["month_in_halfyear"] = ((df[month_col] - 1) % 6 + 1).astype(int)
        df["month_name"] = df[month_col].map({
            1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
            7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December",
        })
        df["month_sin"] = np.sin(2 * np.pi * df[month_col] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df[month_col] / 12)
        df["month_sin_2"] = np.sin(2 * np.pi * 2 * df[month_col] / 12)
        df["month_cos_2"] = np.cos(2 * np.pi * 2 * df[month_col] / 12)
        df["prev_month"] = df[month_col].where(df[month_col] > 1, 13) - 1

        calendar = pd.DataFrame({
            "date": pd.date_range(f"{min(years)}-01-01", f"{max(years)}-12-31", freq="D")
        })
        calendar["year"] = calendar["date"].dt.year
        calendar["month"] = calendar["date"].dt.month
        calendar["weekday"] = calendar["date"].dt.dayofweek
        calendar["is_weekend"] = calendar["weekday"].isin([5, 6]).astype(int)
        calendar["holiday_name"] = calendar["date"].map(holidays)
        calendar["is_holiday"] = calendar["holiday_name"].notna().astype(int)
        calendar["is_additional_working_day"] = calendar["date"].isin(additional_working_days).astype(int)
        calendar["is_nonworking_day"] = (((calendar["is_weekend"] == 1) | (calendar["is_holiday"] == 1)) & (calendar["is_additional_working_day"] == 0)).astype(int)
        calendar["is_working_day"] = 1 - calendar["is_nonworking_day"]
        calendar["is_may_holiday"] = calendar["holiday_name"].isin(["Spring_and_Labour_Holiday", "Victory_Day"]).astype(int)
        calendar["is_single_public_holiday"] = calendar["holiday_name"].isin([
            "Defender_of_the_Fatherland_Day", "International_Womens_Day", "Day_of_Russia", "National_Unity_Day"
        ]).astype(int)
        calendar["is_additional_holiday"] = (calendar["holiday_name"] == "Additional_Holiday").astype(int)
        calendar["is_november_holiday"] = (calendar["holiday_name"] == "National_Unity_Day").astype(int)
        calendar["is_december_additional_holiday"] = ((calendar["month"] == 12) & (calendar["holiday_name"] == "Additional_Holiday")).astype(int)

        month_calendar = (
            calendar.groupby(["year", "month"], as_index=False)
            .agg(
                n_days_in_month=("date", "count"),
                n_weekend_days=("is_weekend", "sum"),
                n_may_holiday_days=("is_may_holiday", "sum"),
                n_single_public_holiday_days=("is_single_public_holiday", "sum"),
                n_november_holiday_days=("is_november_holiday", "sum"),
                n_december_additional_holiday_days=("is_december_additional_holiday", "sum"),
                n_holiday_days=("is_holiday", "sum"),
                n_additional_holiday_days=("is_additional_holiday", "sum"),
                n_working_days=("is_working_day", "sum"),
                n_additional_working_days=("is_additional_working_day", "sum"),
            )
        )
        month_calendar["is_leap_year"] = (
            ((month_calendar["year"] % 4 == 0) & ((month_calendar["year"] % 100 != 0) | (month_calendar["year"] % 400 == 0)))
        ).astype(int)
        month_calendar["holiday_month_type"] = month_calendar["month"].map(
            lambda m: "new_year_holidays" if m == 1 else
            "may_holidays" if m == 5 else
            "single_public_holiday" if m in [2, 3, 6, 11] else
            "pre_new_year" if m == 12 else
            "no_major_holiday"
        )

        df = df.merge(
            month_calendar,
            left_on=[year_col, month_col],
            right_on=["year", "month"],
            how="left",
            suffixes=("", "_calendar"),
        ).drop(columns=["year_calendar", "month_calendar"], errors="ignore")

        df["is_middle_of_quarter"] = (((df[month_col] - 1) % 3 + 1) == 2).astype(int)
        df["is_pre_new_year_month"] = df[month_col].isin([11, 12]).astype(int)
        df["is_pre_may_holidays_month"] = (df[month_col] == 4).astype(int)
        df["is_back_to_school_month"] = df[month_col].isin([8, 9]).astype(int)
        df["is_business_activity_low_month"] = df[month_col].isin([1, 5, 7, 8]).astype(int)
        df["is_business_activity_high_month"] = df[month_col].isin([3, 4, 9, 10, 11]).astype(int)

        return df

    def _add_static(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Traffic
        df["total_traffic_per_hour"] = _col(df, "pedestrian_traffic_per_hour") + _col(df, "car_traffic_per_hour")
        df["pedestrian_traffic_share"] = _safe_divide(_col(df, "pedestrian_traffic_per_hour"), df["total_traffic_per_hour"], fill=0.0)
        df["car_to_pedestrian_traffic_ratio"] = _safe_divide(_col(df, "car_traffic_per_hour"), _col(df, "pedestrian_traffic_per_hour"), fill=0.0)

        for c in ["population_size", "number_of_households", "pedestrian_traffic_per_hour", "car_traffic_per_hour"]:
            if c in df.columns:
                df[f"log_{c}"] = np.log1p(df[c].clip(lower=0).fillna(0))

        if "settlement" in df.columns:
            df["is_moscow"] = df["settlement"].astype(str).str.strip().str.lower().eq("москва г").astype(int)
        else:
            df["is_moscow"] = 0

        df["is_million_plus_city"] = (_col(df, "population_size") >= 1_000_000).astype(int)

        def _city_type(pop, is_moscow):
            if is_moscow == 1:
                return "moscow"
            if pd.isna(pop) or pop <= 0:
                return "unknown_or_zero"
            if pop < 10_000:
                return "small_settlement"
            if pop < 50_000:
                return "small_town"
            if pop < 250_000:
                return "medium_city"
            if pop < 1_000_000:
                return "large_city"
            return "million_plus_city"

        df["population_city_type"] = [_city_type(p, m) for p, m in zip(_col(df, "population_size"), df["is_moscow"])]

        for src, dst in [
            ("schools_300m", "has_school_nearby"),
            ("public_transport_stops_300m", "has_transport_nearby"),
            ("marketplaces_deliveries_parcel_lockers_100m", "has_marketplace_nearby"),
            ("medical_institutions_and_pharmacies_300m", "has_medical_nearby"),
            ("grocery_stores_500m", "has_grocery_competition"),
            ("pyaterochka_stores_500m", "has_pyaterochka_nearby"),
        ]:
            df[dst] = (_col(df, src) > 0).astype(int)

        df["is_high_grocery_competition"] = (_col(df, "grocery_stores_500m") >= 5).astype(int)
        df["is_high_pyaterochka_density"] = (_col(df, "pyaterochka_stores_500m") >= 3).astype(int)

        df["non_pyaterochka_grocery_stores_500m"] = (_col(df, "grocery_stores_500m") - _col(df, "pyaterochka_stores_500m")).clip(lower=0)
        df["pyaterochka_share_among_grocery"] = _safe_divide(_col(df, "pyaterochka_stores_500m"), _col(df, "grocery_stores_500m"), fill=0.0)
        df["non_pyaterochka_competition_share"] = _safe_divide(df["non_pyaterochka_grocery_stores_500m"], _col(df, "grocery_stores_500m"), fill=0.0)
        df["grocery_per_1000_people"] = _safe_divide(_col(df, "grocery_stores_500m") * 1000, _col(df, "population_size"), fill=0.0)
        df["pyaterochka_per_1000_people"] = _safe_divide(_col(df, "pyaterochka_stores_500m") * 1000, _col(df, "population_size"), fill=0.0)
        df["competition_per_1000_households"] = _safe_divide(_col(df, "grocery_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0)
        df["pyaterochka_per_1000_households"] = _safe_divide(_col(df, "pyaterochka_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0)

        df["people_per_household"] = _safe_divide(_col(df, "population_size"), _col(df, "number_of_households"), fill=0.0)
        df["households_per_1000_people"] = _safe_divide(_col(df, "number_of_households") * 1000, _col(df, "population_size"), fill=0.0)

        df["traffic_per_1000_people"] = _safe_divide(df["total_traffic_per_hour"] * 1000, _col(df, "population_size"), fill=0.0)
        df["traffic_per_1000_households"] = _safe_divide(df["total_traffic_per_hour"] * 1000, _col(df, "number_of_households"), fill=0.0)
        df["pedestrian_traffic_per_1000_people"] = _safe_divide(_col(df, "pedestrian_traffic_per_hour") * 1000, _col(df, "population_size"), fill=0.0)

        df["social_infra_score"] = _col(df, "medical_institutions_and_pharmacies_300m") + _col(df, "schools_300m") + _col(df, "public_transport_stops_300m")
        df["commercial_infra_score"] = _col(df, "marketplaces_deliveries_parcel_lockers_100m") + _col(df, "grocery_stores_500m")
        df["poi_score"] = (
            _col(df, "marketplaces_deliveries_parcel_lockers_100m")
            + _col(df, "medical_institutions_and_pharmacies_300m")
            + _col(df, "schools_300m")
            + _col(df, "public_transport_stops_300m")
            + _col(df, "grocery_stores_500m")
        )
        df["poi_per_1000_households"] = _safe_divide(df["poi_score"] * 1000, _col(df, "number_of_households"), fill=0.0)
        df["public_transport_share_in_social_infra"] = _safe_divide(_col(df, "public_transport_stops_300m"), df["social_infra_score"], fill=0.0)
        df["medical_share_in_social_infra"] = _safe_divide(_col(df, "medical_institutions_and_pharmacies_300m"), df["social_infra_score"], fill=0.0)

        df["cash_register_hours_capacity"] = _col(df, "number_of_cash_registers") * _col(df, "working_hours_per_day")
        df["traffic_per_cash_register"] = _safe_divide(df["total_traffic_per_hour"], _col(df, "number_of_cash_registers"), fill=0.0)
        df["traffic_per_working_hour"] = _safe_divide(df["total_traffic_per_hour"], _col(df, "working_hours_per_day"), fill=0.0)
        df["households_per_cash_register"] = _safe_divide(_col(df, "number_of_households"), _col(df, "number_of_cash_registers"), fill=0.0)
        df["population_per_cash_register"] = _safe_divide(_col(df, "population_size"), _col(df, "number_of_cash_registers"), fill=0.0)
        df["cash_registers_per_total_traffic"] = _safe_divide(_col(df, "number_of_cash_registers"), df["total_traffic_per_hour"], fill=0.0)

        if include_current_receipt_features:
            if {"avg_promo_items_per_receipt", "avg_items_per_receipt"}.issubset(df.columns):
                df["promo_items_share"] = _safe_divide(_col(df, "avg_promo_items_per_receipt"), _col(df, "avg_items_per_receipt"), fill=0.0)
            if {"avg_items_per_receipt", "working_hours_per_day"}.issubset(df.columns):
                df["items_x_working_hours"] = _col(df, "avg_items_per_receipt") * _col(df, "working_hours_per_day")

        df["alcohol_x_capacity"] = _col(df, "alcohol_license_flag") * df["cash_register_hours_capacity"]
        df["alcohol_x_total_traffic"] = _col(df, "alcohol_license_flag") * df["total_traffic_per_hour"]
        if include_current_receipt_features and "avg_items_per_receipt" in df.columns:
            df["alcohol_x_items"] = _col(df, "alcohol_license_flag") * _col(df, "avg_items_per_receipt")

        df["traffic_x_total_poi"] = df["total_traffic_per_hour"] * df["poi_score"]
        df["traffic_x_grocery_competition"] = df["total_traffic_per_hour"] * _col(df, "grocery_stores_500m")
        df["traffic_x_pyaterochka_competition"] = df["total_traffic_per_hour"] * _col(df, "pyaterochka_stores_500m")
        df["pedestrian_traffic_x_transport"] = _col(df, "pedestrian_traffic_per_hour") * _col(df, "public_transport_stops_300m")
        if "is_back_to_school_month" in df.columns:
            df["schools_x_back_to_school"] = _col(df, "schools_300m") * _col(df, "is_back_to_school_month")

        # Store-level counts / exogenous group means. They use only exogenous known columns, not target.
        if id_col in df.columns:
            static_cols = [
                id_col, "settlement", "region", "opening_date_category", "trading_area_category",
                "total_traffic_per_hour", "grocery_stores_500m", "pyaterochka_stores_500m",
                "number_of_cash_registers", "poi_score",
            ]
            static_cols = [c for c in static_cols if c in df.columns]
            store_static = df[static_cols].drop_duplicates(subset=[id_col]).copy()

            for group_cols, new_name in [
                (["settlement"], "settlement_store_count"),
                (["region"], "region_store_count"),
                (["region", "trading_area_category"], "region_trading_area_store_count"),
                (["region", "opening_date_category"], "region_opening_age_store_count"),
            ]:
                if all(c in store_static.columns for c in group_cols):
                    cnt = store_static.groupby(group_cols, dropna=False)[id_col].nunique().rename(new_name).reset_index()
                    df = df.merge(cnt, on=group_cols, how="left")

            exog_cols = [c for c in [
                "total_traffic_per_hour", "grocery_stores_500m", "pyaterochka_stores_500m", "number_of_cash_registers", "poi_score"
            ] if c in store_static.columns]

            if exog_cols and "region" in store_static.columns:
                agg = store_static.groupby("region", dropna=False)[exog_cols].mean().add_prefix("region_").add_suffix("_mean").reset_index()
                df = df.merge(agg, on="region", how="left")
            if exog_cols and "settlement" in store_static.columns:
                agg = store_static.groupby("settlement", dropna=False)[exog_cols].mean().add_prefix("settlement_").add_suffix("_mean").reset_index()
                df = df.merge(agg, on="settlement", how="left")

        return df

    def _prepare_base_parts(parts: dict[str, pd.DataFrame]) -> tuple[dict[str, pd.DataFrame], dict]:
        checked = {}
        for name, p in parts.items():
            p = _ensure_time_year_month(p, name)
            _validate_required(p, {id_col, time_col}, name)
            checked[name] = p

        combined = _part_concat(checked)
        combined = _drop_preexisting_generated_cols(combined)
        combined = _ensure_time_year_month(combined, "combined")

        if add_calendar_features:
            combined = _add_calendar(combined)
            combined = _ensure_time_year_month(combined, "combined_after_calendar")

        if add_static_features:
            combined = _add_static(combined)

        parts_out = _split_parts(combined, list(parts.keys()))
        return parts_out, {"base_shape": combined.shape}

    # ============================================================
    # History feature blocks
    # ============================================================

    def _combine_history_future(history_df: pd.DataFrame, future_df: pd.DataFrame, future_name: str):
        history_df = history_df.copy()
        future_df = future_df.copy()
        history_df["__part__"] = "history"
        future_df["__part__"] = future_name
        history_df["__row_order__"] = np.arange(len(history_df))
        future_df["__row_order__"] = np.arange(len(future_df))
        combined = pd.concat([history_df, future_df], axis=0, ignore_index=True, sort=False)
        combined = _ensure_time_year_month(combined, "history_future_combined")
        return combined

    def _restore_pair(combined: pd.DataFrame, future_name: str):
        history_features = (
            combined[combined["__part__"] == "history"]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        future_features = (
            combined[combined["__part__"] == future_name]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        return history_features, future_features

    def _merge_lag_feature(combined, history, source_col, lag, feature_name, by_cols):
        if source_col not in history.columns:
            return combined, False
        lag_df = history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")
        combined = combined.merge(lag_df[by_cols + [time_col, feature_name]], on=by_cols + [time_col], how="left")
        return combined, True

    def _add_target_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_target_lags")
        _validate_unique_panel(history, "history_for_target_lags")

        created = []
        for lag in target_lags:
            fname = f"{target_col}_lag_{lag}"
            combined, ok = _merge_lag_feature(combined, history, target_col, lag, fname, [id_col])
            if ok:
                created.append(fname)
                if target_history_scale in {"log", "both"}:
                    log_name = f"log_{fname}"
                    combined[log_name] = np.log1p(combined[fname].clip(lower=0))
                    created.append(log_name)

        # Rolling features are based only on past history values; future target is ignored even if present.
        if target_rolling_windows:
            tmp = combined[[id_col, time_col, "__part__", target_col]].copy()
            tmp["__target_for_roll__"] = np.where(tmp["__part__"].eq("history"), tmp[target_col], np.nan)
            tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
            shifted = tmp.groupby(id_col, sort=False)["__target_for_roll__"].shift(1)
            for window in target_rolling_windows:
                rg = shifted.groupby(tmp[id_col], sort=False)
                mean_col = f"{target_col}_roll_mean_{window}"
                std_col = f"{target_col}_roll_std_{window}"
                cv_col = f"{target_col}_roll_cv_{window}"
                tmp[mean_col] = rg.rolling(window, min_periods=target_rolling_min_periods).mean().reset_index(level=0, drop=True)
                tmp[std_col] = rg.rolling(window, min_periods=target_rolling_min_periods).std().reset_index(level=0, drop=True)
                tmp[cv_col] = _safe_divide(tmp[std_col], tmp[mean_col], fill=np.nan)
                created.extend([mean_col, std_col, cv_col])
            tmp = tmp.sort_index()
            for c in created:
                if c in tmp.columns and c not in combined.columns:
                    combined[c] = tmp[c]

        def lag_col(lag): return f"{target_col}_lag_{lag}"

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_diff_lag_{a}_{b}"
                combined[c] = combined[lag_col(a)] - combined[lag_col(b)]
                created.append(c)

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_ratio_lag_{a}_{b}"
                combined[c] = _safe_divide(combined[lag_col(a)], combined[lag_col(b)], fill=np.nan)
                created.append(c)

        for window in target_rolling_windows:
            roll_mean = f"{target_col}_roll_mean_{window}"
            if {lag_col(1), roll_mean}.issubset(combined.columns):
                c = f"{target_col}_lag_1_to_roll_mean_{window}"
                combined[c] = _safe_divide(combined[lag_col(1)], combined[roll_mean], fill=np.nan)
                created.append(c)

        if {lag_col(12), f"{target_col}_roll_mean_12"}.issubset(combined.columns):
            c = f"{target_col}_lag_12_to_roll_mean_12"
            combined[c] = _safe_divide(combined[lag_col(12)], combined[f"{target_col}_roll_mean_12"], fill=np.nan)
            created.append(c)

        if {lag_col(12), lag_col(13)}.issubset(combined.columns):
            c1 = f"{target_col}_seasonal_diff_12_13"
            c2 = f"{target_col}_seasonal_ratio_12_13"
            combined[c1] = combined[lag_col(12)] - combined[lag_col(13)]
            combined[c2] = _safe_divide(combined[lag_col(12)], combined[lag_col(13)], fill=np.nan)
            created.extend([c1, c2])

        if target_history_scale in {"log", "both"} and add_log_target_dynamics:
            for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
                la, lb = f"log_{lag_col(a)}", f"log_{lag_col(b)}"
                if {la, lb}.issubset(combined.columns):
                    c = f"log_{target_col}_diff_lag_{a}_{b}"
                    combined[c] = combined[la] - combined[lb]
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_category_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_category_lags")

        specs = group_specs
        if specs is None:
            specs = [
                "region",
                "settlement",
                ["region", "trading_area_category"],
                ["region", "opening_date_category"],
                ["settlement", "trading_area_category"],
            ]
        norm_specs = []
        for spec in specs:
            spec = [spec] if isinstance(spec, str) else list(spec)
            if all(c in combined.columns for c in spec):
                norm_specs.append(spec)
        if not norm_specs:
            return (*_restore_pair(combined, future_name), [])

        created = []
        internal_store_lag = None
        store_lag_1 = f"{target_col}_lag_1"
        if store_lag_1 not in combined.columns:
            internal_store_lag = f"__{target_col}_store_lag_1__"
            combined, _ = _merge_lag_feature(combined, history, target_col, 1, internal_store_lag, [id_col])
            store_lag_1 = internal_store_lag

        for group_cols in norm_specs:
            prefix = "__".join(group_cols)
            agg_col = f"__{prefix}_{target_col}_{group_agg_stat}__"
            monthly_agg = (
                history.groupby(group_cols + [time_col], dropna=False)[target_col]
                .agg(group_agg_stat)
                .reset_index()
                .rename(columns={target_col: agg_col})
            )
            monthly_agg[time_col] = monthly_agg[time_col].astype(int) + int(group_lag)
            group_lag_col = f"{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
            monthly_agg = monthly_agg.rename(columns={agg_col: group_lag_col})
            combined = combined.merge(monthly_agg[group_cols + [time_col, group_lag_col]], on=group_cols + [time_col], how="left")

            if add_group_lag_feature:
                created.append(group_lag_col)

            if add_group_relative_ratio:
                c = f"{target_col}_lag_1_to_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = _safe_divide(combined[store_lag_1], combined[group_lag_col], fill=np.nan)
                created.append(c)

            if add_group_relative_diff:
                c = f"{target_col}_lag_1_minus_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = combined[store_lag_1] - combined[group_lag_col]
                created.append(c)

        if internal_store_lag is not None:
            combined = combined.drop(columns=[internal_store_lag], errors="ignore")

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_monthly_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col}, "history_for_monthly_lags")
        _validate_unique_panel(history, "history_for_monthly_lags")

        existing_monthly = [c for c in current_monthly_cols if c in history.columns]
        if not existing_monthly:
            return (*_restore_pair(combined, future_name), [])

        if monthly_feature_config is None:
            cfg_all = {
                "avg_promo_items_per_receipt": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "avg_items_per_receipt": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "avg_cancellations": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("mean", "std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "working_hours_per_day": {
                    "lags": (1,), "rolling_windows": (), "rolling_stats": (), "diff_pairs": (), "ratio_pairs": (),
                },
            }
        else:
            cfg_all = monthly_feature_config

        created = []

        for colname in existing_monthly:
            cfg = cfg_all.get(colname, {
                "lags": (1, 3, 12), "rolling_windows": (3, 12),
                "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
            })

            # Time-aware lags.
            for lag in tuple(cfg.get("lags", ())):
                c = f"{colname}_lag_{lag}"
                combined, ok = _merge_lag_feature(combined, history, colname, lag, c, [id_col])
                if ok:
                    created.append(c)
                    if monthly_history_scale in {"log", "both"}:
                        lc = f"log_{colname}_lag_{lag}"
                        combined[lc] = np.log1p(combined[c].clip(lower=0))
                        created.append(lc)

            # Rolling over past observed history.
            windows = tuple(cfg.get("rolling_windows", ()))
            stats = tuple(cfg.get("rolling_stats", ()))
            if windows and stats:
                tmp = combined[[id_col, time_col, "__part__", colname]].copy()
                tmp["__x__"] = np.where(tmp["__part__"].eq("history"), tmp[colname], np.nan)
                tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
                shifted = tmp.groupby(id_col, sort=False)["__x__"].shift(1)
                for window in windows:
                    rg = shifted.groupby(tmp[id_col], sort=False)
                    mean_s = std_s = min_s = max_s = None
                    if "mean" in stats or "cv" in stats:
                        mean_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).mean().reset_index(level=0, drop=True)
                        if "mean" in stats:
                            c = f"{colname}_roll_mean_{window}"
                            tmp[c] = mean_s
                            created.append(c)
                    if "std" in stats or "cv" in stats:
                        std_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).std().reset_index(level=0, drop=True)
                        if "std" in stats:
                            c = f"{colname}_roll_std_{window}"
                            tmp[c] = std_s
                            created.append(c)
                    if "min" in stats or "range" in stats:
                        min_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).min().reset_index(level=0, drop=True)
                        if "min" in stats:
                            c = f"{colname}_roll_min_{window}"
                            tmp[c] = min_s
                            created.append(c)
                    if "max" in stats or "range" in stats:
                        max_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).max().reset_index(level=0, drop=True)
                        if "max" in stats:
                            c = f"{colname}_roll_max_{window}"
                            tmp[c] = max_s
                            created.append(c)
                    if "range" in stats:
                        c = f"{colname}_roll_range_{window}"
                        tmp[c] = max_s - min_s
                        created.append(c)
                    if "cv" in stats:
                        c = f"{colname}_roll_cv_{window}"
                        tmp[c] = _safe_divide(std_s, mean_s, fill=np.nan)
                        created.append(c)
                tmp = tmp.sort_index()
                for c in created:
                    if c in tmp.columns and c not in combined.columns:
                        combined[c] = tmp[c]

            for a, b in tuple(cfg.get("diff_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_diff_lag_{a}_{b}"
                    combined[c] = combined[ca] - combined[cb]
                    created.append(c)

            for a, b in tuple(cfg.get("ratio_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_ratio_lag_{a}_{b}"
                    combined[c] = _safe_divide(combined[ca], combined[cb], fill=np.nan)
                    created.append(c)

        if add_monthly_cross_lags:
            for lag in monthly_cross_lags:
                promo = f"avg_promo_items_per_receipt_lag_{lag}"
                items = f"avg_items_per_receipt_lag_{lag}"
                canc = f"avg_cancellations_lag_{lag}"
                hours = f"working_hours_per_day_lag_{lag}"

                if "promo_share" in monthly_cross_feature_types and {promo, items}.issubset(combined.columns):
                    c = f"promo_items_share_lag_{lag}"
                    combined[c] = _safe_divide(combined[promo], combined[items], fill=np.nan)
                    created.append(c)
                if "promo_x_items" in monthly_cross_feature_types and {promo, items}.issubset(combined.columns):
                    c = f"promo_x_items_lag_{lag}"
                    combined[c] = combined[promo] * combined[items]
                    created.append(c)
                if "items_x_hours" in monthly_cross_feature_types and {items, hours}.issubset(combined.columns):
                    c = f"items_x_working_hours_lag_{lag}"
                    combined[c] = combined[items] * combined[hours]
                    created.append(c)
                if "cancellations_per_item" in monthly_cross_feature_types and {canc, items}.issubset(combined.columns):
                    c = f"cancellations_per_item_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[items], fill=np.nan)
                    created.append(c)
                if "cancellations_per_hour" in monthly_cross_feature_types and {canc, hours}.issubset(combined.columns):
                    c = f"cancellations_per_working_hour_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[hours], fill=np.nan)
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)

        if drop_current_monthly_cols:
            combined = combined.drop(columns=list(existing_monthly), errors="ignore")
        if drop_current_monthly_derived_cols:
            combined = combined.drop(columns=list(current_monthly_derived_cols), errors="ignore")

        return (*_restore_pair(combined, future_name), created)

    # ============================================================
    # Final cleanup
    # ============================================================

    def _future_unknown_cols():
        cols = []
        if drop_current_monthly_cols:
            cols.extend(list(current_monthly_cols))
        if drop_current_monthly_derived_cols:
            cols.extend(list(current_monthly_derived_cols))
        if not working_hours_future_known:
            cols.extend([
                "working_hours_per_day",
                "cash_register_hours_capacity",
                "traffic_per_working_hour",
                "alcohol_x_capacity",
                "items_x_working_hours",
            ])
        if drop_year_col:
            cols.append(year_col)
        return sorted(set(cols))

    def _obvious_duplicate_cols():
        cols = [
            f"{target_col}_roll_mean_3", f"{target_col}_roll_mean_12",
            f"{target_col}_lag_13", f"{target_col}_diff_lag_3_6", f"{target_col}_ratio_lag_3_6",
            "avg_items_per_receipt_diff_lag_1_12",
            "avg_cancellations_roll_mean_3", "avg_cancellations_roll_mean_12",
            "cancellations_per_item_lag_1", "cancellations_per_item_lag_3", "cancellations_per_item_lag_12",
            "settlement_store_count", "total_traffic_per_hour", "non_pyaterochka_grocery_stores_500m",
        ]

        if target_history_scale == "raw":
            cols.extend([f"log_{target_col}_lag_{l}" for l in target_lags])
            cols.extend([c for c in []])
        elif target_history_scale == "log":
            raw_lags = [f"{target_col}_lag_{l}" for l in target_lags if l != 1]
            if not keep_raw_pto_lag_1_anchor_for_log:
                raw_lags.append(f"{target_col}_lag_1")
            cols.extend(raw_lags)
            cols.extend([
                f"{target_col}_diff_lag_1_2", f"{target_col}_diff_lag_1_3",
                f"{target_col}_diff_lag_1_6", f"{target_col}_diff_lag_1_12",
                f"{target_col}_diff_lag_3_6", f"{target_col}_ratio_lag_1_2",
                f"{target_col}_ratio_lag_1_3", f"{target_col}_ratio_lag_1_6",
                f"{target_col}_ratio_lag_1_12", f"{target_col}_ratio_lag_3_6",
                f"{target_col}_ratio_lag_6_12", f"{target_col}_seasonal_diff_12_13",
                f"{target_col}_seasonal_ratio_12_13",
            ])

        if monthly_history_scale == "raw":
            for c in current_monthly_cols:
                cols.extend([f"log_{c}_lag_1", f"log_{c}_lag_3", f"log_{c}_lag_12"])
        elif monthly_history_scale == "log":
            for c in current_monthly_cols:
                cols.extend([f"{c}_lag_3", f"{c}_lag_12"])

        if extra_drop_cols is not None:
            cols.extend(list(extra_drop_cols))
        return sorted(set(cols))

    def _apply_final_clean(train_part, future_part):
        train_part = train_part.copy()
        future_part = future_part.copy()
        drop_cols = _future_unknown_cols()
        if drop_obvious_duplicates:
            drop_cols.extend(_obvious_duplicate_cols())
        drop_cols = sorted(set(c for c in drop_cols if c in train_part.columns or c in future_part.columns))
        train_part = train_part.drop(columns=drop_cols, errors="ignore")
        future_part = future_part.drop(columns=drop_cols, errors="ignore")

        # Replace infinities only; do not fill all lag NaNs unless requested.
        for df_ in (train_part, future_part):
            num_cols = df_.select_dtypes(include=[np.number]).columns
            df_[num_cols] = df_[num_cols].replace([np.inf, -np.inf], np.nan)
            if fill_history_na:
                df_[num_cols] = df_[num_cols].fillna(fill_value)

        # Align columns in stable order.
        ordered_cols = list(train_part.columns)
        for c in future_part.columns:
            if c not in ordered_cols:
                ordered_cols.append(c)
        for c in ordered_cols:
            if c not in train_part.columns:
                train_part[c] = np.nan
            if c not in future_part.columns:
                future_part[c] = np.nan
        train_part = train_part[ordered_cols]
        future_part = future_part[ordered_cols]
        return train_part, future_part, drop_cols

    def _downcast(df):
        if not downcast_float_features:
            return df
        df = df.copy()
        float_cols = [c for c in df.select_dtypes(include=["float64"]).columns if c != target_col]
        if float_cols:
            df[float_cols] = df[float_cols].astype("float32")
        return df

    def _generate_pair(history_base, future_base, future_name):
        info = {}
        h, f = history_base.copy(), future_base.copy()

        if add_target_history:
            h, f, created = _add_target_history_pair(h, f, future_name)
            info["target_history_features"] = created
        else:
            info["target_history_features"] = []

        if add_category_target_history:
            h, f, created = _add_category_history_pair(h, f, future_name)
            info["category_target_history_features"] = created
        else:
            info["category_target_history_features"] = []

        if add_non_target_monthly_history:
            h, f, created = _add_monthly_history_pair(h, f, future_name)
            info["non_target_monthly_history_features"] = created
        else:
            info["non_target_monthly_history_features"] = []
            if drop_current_monthly_cols:
                h = h.drop(columns=[c for c in current_monthly_cols if c in h.columns], errors="ignore")
                f = f.drop(columns=[c for c in current_monthly_cols if c in f.columns], errors="ignore")
            if drop_current_monthly_derived_cols:
                h = h.drop(columns=[c for c in current_monthly_derived_cols if c in h.columns], errors="ignore")
                f = f.drop(columns=[c for c in current_monthly_derived_cols if c in f.columns], errors="ignore")

        h, f, dropped = _apply_final_clean(h, f)
        info["dropped_final_cols"] = dropped
        h = _downcast(h)
        f = _downcast(f)
        return h, f, info

    # ============================================================
    # Main flow
    # ============================================================

    train_df = _ensure_time_year_month(train_df, "train_df")
    _validate_required(train_df, {id_col, time_col, target_col}, "train_df")
    _validate_unique_panel(train_df, "train_df")

    if make_val_from_train:
        if val_df is not None:
            raise ValueError("Нельзя одновременно передать val_df и make_val_from_train=True.")
        if val_time_idx is None:
            val_time_idx = int(train_df[time_col].max())
        val_df = train_df[train_df[time_col] == val_time_idx].copy()
        train_df = train_df[train_df[time_col] < val_time_idx].copy()
        if verbose:
            print(f"Auto val_time_idx: {val_time_idx}")
            print("Auto train shape:", train_df.shape)
            print("Auto val shape:", val_df.shape)

    if val_df is not None:
        val_df = _ensure_time_year_month(val_df, "val_df")
        _validate_required(val_df, {id_col, time_col}, "val_df")
        _validate_unique_panel(val_df, "val_df")

    if test_df is not None:
        test_df = _ensure_time_year_month(test_df, "test_df")
        _validate_required(test_df, {id_col, time_col}, "test_df")
        _validate_unique_panel(test_df, "test_df")
        if target_col not in test_df.columns:
            test_df[target_col] = np.nan

    output = {
        "train_features": None,
        "val_features": None,
        "test_train_features": None,
        "test_features": None,
        "feature_cols": None,
        "categorical_cols": None,
        "feature_info": {},
    }

    if val_df is not None:
        base_parts, base_info = _prepare_base_parts({"train": train_df, "val": val_df})
        train_features, val_features, val_info = _generate_pair(base_parts["train"], base_parts["val"], "val")
        output["train_features"] = train_features
        output["val_features"] = val_features
        output["feature_info"]["val"] = {**base_info, **val_info}
        if verbose:
            print("Generated train_features:", train_features.shape)
            print("Generated val_features:", val_features.shape)

    if test_df is not None:
        if val_df is not None and use_val_as_test_history:
            test_history = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
        else:
            test_history = train_df.copy()
        test_history = _ensure_time_year_month(test_history, "test_history")
        _validate_unique_panel(test_history, "test_history")

        base_parts, base_info = _prepare_base_parts({"train": test_history, "test": test_df})
        test_train_features, test_features, test_info = _generate_pair(base_parts["train"], base_parts["test"], "test")
        output["test_train_features"] = test_train_features
        output["test_features"] = test_features
        output["feature_info"]["test"] = {**base_info, **test_info}
        if verbose:
            print("Generated test_train_features:", test_train_features.shape)
            print("Generated test_features:", test_features.shape)

    # Select columns for model.
    ref = output["test_train_features"] if output["test_train_features"] is not None else output["train_features"]
    if ref is not None:
        exclude = {target_col}
        feature_cols = [c for c in ref.columns if c not in exclude]
        categorical_cols = [
            c for c in feature_cols
            if c in ref.columns and (pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c]))
        ]
        output["feature_cols"] = feature_cols
        output["categorical_cols"] = categorical_cols

    gc.collect()
    return output


def _ensure_time_year_month_external(
    df: pd.DataFrame,
    *,
    year_col: str,
    month_col: str,
    time_col: str,
    min_year: int,
    name: str = "df",
) -> pd.DataFrame:
    """Внешний helper для wrapper/enhancer: гарантирует year/month/time."""
    out = df.copy()
    has_year_month = year_col in out.columns and month_col in out.columns
    has_time = time_col in out.columns

    if not has_year_month and not has_time:
        raise ValueError(f"В {name} нет ни year/month, ни time_col='{time_col}'.")

    if has_year_month:
        out[year_col] = pd.to_numeric(out[year_col], errors="raise").astype(int)
        out[month_col] = pd.to_numeric(out[month_col], errors="raise").astype(int)
        out[time_col] = ((out[year_col] - int(min_year)) * 12 + (out[month_col] - 1)).astype(int)
    else:
        out[time_col] = pd.to_numeric(out[time_col], errors="raise").astype(int)
        out[year_col] = (int(min_year) + out[time_col] // 12).astype(int)
        out[month_col] = (out[time_col] % 12 + 1).astype(int)

    return out


def _calendar_month_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (time_s.astype(int) % 12 + 1).astype(int)


def _year_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (int(min_year) + time_s.astype(int) // 12).astype(int)


def _days_in_month_from_year_month(year_s: pd.Series, month_s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(
        {
            "year": pd.to_numeric(year_s, errors="coerce").astype("Int64"),
            "month": pd.to_numeric(month_s, errors="coerce").astype("Int64"),
            "day": 1,
        },
        errors="coerce",
    )
    return dt.dt.days_in_month.astype("float32")


def _safe_divide_external(a, b, fill=np.nan):
    a = a if isinstance(a, pd.Series) else pd.Series(a)
    b = b if isinstance(b, pd.Series) else pd.Series(b)
    out = a.astype(float) / b.replace(0, np.nan).astype(float)
    out = out.replace([np.inf, -np.inf], np.nan)
    if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
        out = out.fillna(fill)
    return out


def _safe_log_external(x, eps: float = 1e-9):
    return np.log(np.clip(pd.Series(x).astype(float), eps, None))



def _assign_lag1_rto_size_bin(
    df: pd.DataFrame,
    *,
    time_col: str,
    lag1_col: str,
    out_col: str = "lag1_rto_size_bin",
    n_bins: int = 5,
) -> pd.Series:
    """
    Cross-sectional size bin by pto_lag_1 inside each forecast month.
    Uses only lagged target values, therefore it is safe for future rows.
    """
    result = pd.Series(np.nan, index=df.index, dtype="object")

    if lag1_col not in df.columns:
        return result

    for _, idx in df.groupby(time_col, sort=False).groups.items():
        s = pd.to_numeric(df.loc[idx, lag1_col], errors="coerce")
        valid = s.notna()
        if valid.sum() == 0:
            continue
        if valid.sum() == 1 or s[valid].nunique(dropna=True) == 1:
            result.loc[s[valid].index] = "q3"
            continue
        q = int(min(n_bins, valid.sum(), s[valid].nunique(dropna=True)))
        if q < 2:
            result.loc[s[valid].index] = "q3"
            continue
        try:
            codes = pd.qcut(
                s[valid].rank(method="first"),
                q=q,
                labels=False,
                duplicates="drop",
            )
            result.loc[s[valid].index] = [f"q{int(x) + 1}" if pd.notna(x) else np.nan for x in codes]
        except ValueError:
            result.loc[s[valid].index] = "q3"

    return result


def _expanding_median_factor_from_history(
    combined: pd.DataFrame,
    observed: pd.DataFrame,
    *,
    group_cols: list[str],
    time_col: str,
    obs_col: str,
    out_col: str,
) -> pd.Series:
    """
    For every row in combined returns median(obs_col) over historical observations
    with the same group key and strictly earlier time_col.

    Implementation detail: query rows are sorted before observation rows at the
    same time index, so same-month target is never used for the same row/month.
    """
    keys = list(group_cols) + [time_col]
    result = pd.Series(np.nan, index=combined.index, dtype="float64")

    if not group_cols or obs_col not in observed.columns:
        return result
    if any(c not in combined.columns for c in keys) or any(c not in observed.columns for c in keys):
        return result

    obs = observed[keys + [obs_col]].dropna(subset=[obs_col]).copy()
    if obs.empty:
        return result

    obs_gt = (
        obs.groupby(keys, dropna=False, as_index=False)[obs_col]
        .median()
        .rename(columns={obs_col: "__obs_factor__"})
    )
    if obs_gt.empty:
        return result

    queries = combined[keys].copy()
    queries["__query_index__"] = combined.index
    queries["__obs_factor__"] = np.nan
    queries["__is_query__"] = 1
    queries["__order__"] = 0

    obs_gt["__query_index__"] = -1
    obs_gt["__is_query__"] = 0
    obs_gt["__order__"] = 1

    stack = pd.concat([queries, obs_gt], ignore_index=True, sort=False)
    stack = stack.sort_values(group_cols + [time_col, "__order__"], kind="mergesort")

    stack[out_col] = (
        stack.groupby(group_cols, dropna=False)["__obs_factor__"]
        .transform(lambda s: s.expanding(min_periods=1).median())
    )

    q = stack[stack["__is_query__"].eq(1)][["__query_index__", out_col]].copy()
    q = q.drop_duplicates("__query_index__", keep="last")
    result.loc[q["__query_index__"].astype(int).values] = q[out_col].values
    return result


def _add_expanding_calendar_heuristic_features(
    combined: pd.DataFrame,
    *,
    id_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
    eps: float = 1e-9,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Adds the old expanding calendar heuristic block without target leakage.

    Exposed features:
    - lag1_rto_size_bin;
    - same_month_calendar_factor;
    - same_month_region_calendar_factor;
    - same_month_population_type_calendar_factor;
    - same_month_lag1_size_calendar_factor;
    - calendar_growth_multiplier;
    - calendar_heuristic_*_prediction;
    - log_calendar_heuristic_prediction;
    - log_pto_lag_1_to_calendar_heuristic;
    - log_pto_lag_1_to_day_scaled_lag1.

    Internal observed_calendar_factor is deliberately not returned as a model
    feature, because it uses current target on history rows.
    """
    out = combined.copy()
    created: list[str] = []

    def mark(col: str):
        if col in out.columns and col not in created:
            created.append(col)

    pto_lag_1 = f"{target_col}_lag_1"
    required = {"__part__", time_col, target_col, pto_lag_1, "day_scaled_lag1_prediction", "month_day_ratio_to_previous_month"}
    if not required.issubset(out.columns):
        return out, created

    if "calendar_month_num" not in out.columns:
        out["calendar_month_num"] = _calendar_month_from_time(out[time_col], min_year)
        mark("calendar_month_num")

    out["lag1_rto_size_bin"] = _assign_lag1_rto_size_bin(
        out,
        time_col=time_col,
        lag1_col=pto_lag_1,
        out_col="lag1_rto_size_bin",
        n_bins=5,
    )
    mark("lag1_rto_size_bin")

    # Useful log-ratio against the pure day-scaled anchor.
    if f"log_{target_col}_lag_1" not in out.columns:
        out[f"log_{target_col}_lag_1"] = _safe_log_external(out[pto_lag_1], eps=eps)
        mark(f"log_{target_col}_lag_1")
    if "log_day_scaled_lag1_prediction" not in out.columns:
        out["log_day_scaled_lag1_prediction"] = _safe_log_external(out["day_scaled_lag1_prediction"], eps=eps)
        mark("log_day_scaled_lag1_prediction")

    out["log_pto_lag_1_to_day_scaled_lag1"] = (
        out[f"log_{target_col}_lag_1"] - out["log_day_scaled_lag1_prediction"]
    )
    mark("log_pto_lag_1_to_day_scaled_lag1")

    hist_mask = out["__part__"].eq("history") & out[target_col].notna()
    observed = out.loc[hist_mask].copy()
    observed["__observed_calendar_factor__"] = _safe_divide_external(
        observed[target_col],
        observed["day_scaled_lag1_prediction"],
        fill=np.nan,
    )
    observed["__observed_calendar_factor__"] = observed["__observed_calendar_factor__"].replace(
        [np.inf, -np.inf], np.nan
    )
    observed = observed.dropna(subset=["__observed_calendar_factor__"])

    if observed.empty:
        return out, created

    factor_specs = [
        (["calendar_month_num"], "same_month_calendar_factor", None),
    ]
    if "region" in out.columns:
        factor_specs.append((["calendar_month_num", "region"], "same_month_region_calendar_factor", "same_month_calendar_factor"))
    if "population_city_type" in out.columns:
        factor_specs.append((["calendar_month_num", "population_city_type"], "same_month_population_type_calendar_factor", "same_month_calendar_factor"))
    if "lag1_rto_size_bin" in out.columns:
        factor_specs.append((["calendar_month_num", "lag1_rto_size_bin"], "same_month_lag1_size_calendar_factor", "same_month_calendar_factor"))

    for group_cols, factor_col, fallback_col in factor_specs:
        if any(c not in out.columns for c in group_cols):
            continue
        factor = _expanding_median_factor_from_history(
            out,
            observed,
            group_cols=group_cols,
            time_col=time_col,
            obs_col="__observed_calendar_factor__",
            out_col=factor_col,
        )
        out[factor_col] = factor
        if fallback_col is not None and fallback_col in out.columns:
            out[factor_col] = out[factor_col].fillna(out[fallback_col])
        mark(factor_col)

    if "same_month_calendar_factor" in out.columns:
        out["calendar_growth_multiplier"] = (
            out["month_day_ratio_to_previous_month"] * out["same_month_calendar_factor"]
        )
        out["calendar_heuristic_prediction"] = out[pto_lag_1] * out["calendar_growth_multiplier"]
        out["log_calendar_heuristic_prediction"] = _safe_log_external(
            out["calendar_heuristic_prediction"], eps=eps
        )
        out["log_pto_lag_1_to_calendar_heuristic"] = (
            out[f"log_{target_col}_lag_1"] - out["log_calendar_heuristic_prediction"]
        )
        mark("calendar_growth_multiplier")
        mark("calendar_heuristic_prediction")
        mark("log_calendar_heuristic_prediction")
        mark("log_pto_lag_1_to_calendar_heuristic")

    pred_specs = [
        ("same_month_region_calendar_factor", "calendar_heuristic_region_prediction"),
        ("same_month_population_type_calendar_factor", "calendar_heuristic_population_type_prediction"),
        ("same_month_lag1_size_calendar_factor", "calendar_heuristic_lag1_size_prediction"),
    ]
    for factor_col, pred_col in pred_specs:
        if factor_col not in out.columns:
            continue
        out[pred_col] = out[pto_lag_1] * out["month_day_ratio_to_previous_month"] * out[factor_col]
        out[f"log_{pred_col}"] = _safe_log_external(out[pred_col], eps=eps)
        mark(pred_col)
        mark(f"log_{pred_col}")

    created = sorted(set(c for c in created if c in out.columns))
    out[created] = out[created].replace([np.inf, -np.inf], np.nan)
    return out, created

def _add_integrated_anchor_features_pair(
    history_features: pd.DataFrame,
    future_features: pd.DataFrame,
    history_raw: pd.DataFrame,
    future_raw: pd.DataFrame,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Добавляет идеи из build_lag1_features_for_pipeline,
    build_anchor_features_for_pipeline и build_march_hierarchy_ops_features_for_pipeline
    к уже собранным leak-free features.

    Важная логика:
    - лаги target и monthly-cols строятся time-aware merge'ем из history_raw;
    - current target future_features не используется;
    - мартовские коэффициенты 2024 считаются только по history_raw;
    - expanding calendar heuristic считается только по history rows и строго прошлым time_idx;
    - мартовские коэффициенты доступны только для строк после марта 2024, чтобы не
      протаскивать target марта 2024 в более ранние train-строки.
    """
    h_feat = history_features.copy()
    f_feat = future_features.copy()

    h_feat["__part__"] = "history"
    f_feat["__part__"] = "future"
    h_feat["__row_order__"] = np.arange(len(h_feat))
    f_feat["__row_order__"] = np.arange(len(f_feat))

    combined = pd.concat([h_feat, f_feat], ignore_index=True, sort=False)
    combined = _ensure_time_year_month_external(
        combined,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="combined_features",
    )

    history_raw = _ensure_time_year_month_external(
        history_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="history_raw",
    )
    future_raw = _ensure_time_year_month_external(
        future_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="future_raw",
    )

    if id_col not in combined.columns:
        raise ValueError(f"В features нет id_col='{id_col}'.")
    if target_col not in history_raw.columns:
        raise ValueError(f"В history_raw нет target_col='{target_col}'.")

    created: list[str] = []

    def _mark(col):
        if col in combined.columns and col not in created:
            created.append(col)

    def _merge_history_lag(source_history, source_col, lag, feature_name, by_cols):
        nonlocal combined
        if source_col not in source_history.columns:
            return False

        lag_df = source_history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")

        combined = combined.drop(columns=[feature_name], errors="ignore")
        combined = combined.merge(
            lag_df[by_cols + [time_col, feature_name]],
            on=by_cols + [time_col],
            how="left",
        )
        _mark(feature_name)
        return True

    # ------------------------------------------------------------
    # 1. Calendar day features + target lags/daily lags
    # ------------------------------------------------------------
    combined["calendar_month_num"] = _calendar_month_from_time(combined[time_col], min_year)
    combined["calendar_year"] = _year_from_time(combined[time_col], min_year)
    combined["days_in_month"] = _days_in_month_from_year_month(
        combined["calendar_year"], combined["calendar_month_num"]
    )
    _mark("calendar_month_num")
    _mark("days_in_month")

    for lag in target_lags:
        lag = int(lag)
        pto_lag_col = f"{target_col}_lag_{lag}"
        _merge_history_lag(history_raw, target_col, lag, pto_lag_col, [id_col])

        lag_time = combined[time_col].astype(int) - lag
        lag_year = _year_from_time(lag_time, min_year)
        lag_month = _calendar_month_from_time(lag_time, min_year)
        days_lag_col = f"days_lag_{lag}"
        combined[days_lag_col] = _days_in_month_from_year_month(lag_year, lag_month)
        _mark(days_lag_col)

        daily_col = f"daily_{target_col}_lag_{lag}"
        combined[daily_col] = _safe_divide_external(
            combined[pto_lag_col],
            combined[days_lag_col],
            fill=np.nan,
        )
        _mark(daily_col)

        log_pto_col = f"log_{target_col}_lag_{lag}"
        log_daily_col = f"log_daily_{target_col}_lag_{lag}"
        combined[log_pto_col] = _safe_log_external(combined[pto_lag_col], eps=eps)
        combined[log_daily_col] = _safe_log_external(combined[daily_col], eps=eps)
        _mark(log_pto_col)
        _mark(log_daily_col)

    if "days_lag_1" in combined.columns:
        combined["days_in_prev_month"] = combined["days_lag_1"]
        _mark("days_in_prev_month")

    if add_calendar_anchor_features and {f"{target_col}_lag_1", "days_in_month", "days_in_prev_month"}.issubset(combined.columns):
        combined["month_day_ratio_to_previous_month"] = _safe_divide_external(
            combined["days_in_month"],
            combined["days_in_prev_month"],
            fill=np.nan,
        )
        combined["day_scaled_lag1_prediction"] = (
            combined[f"{target_col}_lag_1"] * combined["month_day_ratio_to_previous_month"]
        )
        combined["log_day_scaled_lag1_prediction"] = _safe_log_external(
            combined["day_scaled_lag1_prediction"], eps=eps
        )
        _mark("month_day_ratio_to_previous_month")
        _mark("day_scaled_lag1_prediction")
        _mark("log_day_scaled_lag1_prediction")

        for lag in [2, 3, 6, 12]:
            daily_col = f"daily_{target_col}_lag_{lag}"
            pred_col = f"day_scaled_lag{lag}_prediction"
            if daily_col in combined.columns:
                combined[pred_col] = combined[daily_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

    if add_calendar_anchor_features and add_expanding_calendar_heuristic_features:
        combined, calendar_created = _add_expanding_calendar_heuristic_features(
            combined,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
            eps=eps,
        )
        for c in calendar_created:
            _mark(c)

    # ------------------------------------------------------------
    # 2. Recent daily baselines + trend anchors + same-transition anchors
    # ------------------------------------------------------------
    if add_daily_anchor_features:
        daily_1 = f"daily_{target_col}_lag_1"
        daily_2 = f"daily_{target_col}_lag_2"
        daily_3 = f"daily_{target_col}_lag_3"

        if {daily_1, daily_2}.issubset(combined.columns):
            combined["recent_daily_mean_2"] = combined[[daily_1, daily_2]].mean(axis=1)
            combined["recent_daily_mean_2_prediction"] = combined["recent_daily_mean_2"] * combined["days_in_month"]
            combined["log_recent_daily_mean_2_prediction"] = _safe_log_external(
                combined["recent_daily_mean_2_prediction"], eps=eps
            )
            _mark("recent_daily_mean_2")
            _mark("recent_daily_mean_2_prediction")
            _mark("log_recent_daily_mean_2_prediction")

        if {daily_1, daily_2, daily_3}.issubset(combined.columns):
            recent_cols = [daily_1, daily_2, daily_3]
            combined["recent_daily_mean_3"] = combined[recent_cols].mean(axis=1)
            combined["recent_daily_median_3"] = combined[recent_cols].median(axis=1)
            combined["recent_daily_weighted_123"] = (
                0.60 * combined[daily_1]
                + 0.25 * combined[daily_2]
                + 0.15 * combined[daily_3]
            )

            for col in ["recent_daily_mean_3", "recent_daily_median_3", "recent_daily_weighted_123"]:
                pred_col = f"{col}_prediction"
                combined[pred_col] = combined[col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            combined["daily_growth_1_2"] = _safe_divide_external(combined[daily_1], combined[daily_2], fill=np.nan)
            combined["daily_growth_1_3"] = _safe_divide_external(combined[daily_1], combined[daily_3], fill=np.nan)
            combined["daily_growth_1_2_clipped"] = combined["daily_growth_1_2"].clip(0.70, 1.35)
            combined["daily_growth_1_3_clipped"] = combined["daily_growth_1_3"].clip(0.70, 1.35)
            _mark("daily_growth_1_2")
            _mark("daily_growth_1_3")
            _mark("daily_growth_1_2_clipped")
            _mark("daily_growth_1_3_clipped")

            if "day_scaled_lag1_prediction" in combined.columns:
                combined["trend_day_scaled_lag1_prediction"] = (
                    combined["day_scaled_lag1_prediction"] * combined["daily_growth_1_2_clipped"]
                )
                combined["trend_day_scaled_lag1_prediction_smooth"] = (
                    combined["day_scaled_lag1_prediction"] * np.sqrt(combined["daily_growth_1_2_clipped"])
                )
                combined["log_trend_day_scaled_lag1_prediction"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction"], eps=eps
                )
                combined["log_trend_day_scaled_lag1_prediction_smooth"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction_smooth"], eps=eps
                )
                _mark("trend_day_scaled_lag1_prediction")
                _mark("trend_day_scaled_lag1_prediction_smooth")
                _mark("log_trend_day_scaled_lag1_prediction")
                _mark("log_trend_day_scaled_lag1_prediction_smooth")

    if add_transition_anchor_features:
        pto12, pto13 = f"{target_col}_lag_12", f"{target_col}_lag_13"
        daily1, daily12, daily13 = (
            f"daily_{target_col}_lag_1",
            f"daily_{target_col}_lag_12",
            f"daily_{target_col}_lag_13",
        )

        if {pto12, pto13, f"{target_col}_lag_1"}.issubset(combined.columns):
            combined["same_transition_last_year_factor"] = _safe_divide_external(
                combined[pto12], combined[pto13], fill=np.nan
            )
            combined["same_transition_last_year_factor_clipped"] = (
                combined["same_transition_last_year_factor"].clip(0.75, 1.45)
            )
            combined["same_transition_last_year_prediction"] = (
                combined[f"{target_col}_lag_1"] * combined["same_transition_last_year_factor_clipped"]
            )
            combined["log_same_transition_last_year_prediction"] = _safe_log_external(
                combined["same_transition_last_year_prediction"], eps=eps
            )
            _mark("same_transition_last_year_factor")
            _mark("same_transition_last_year_factor_clipped")
            _mark("same_transition_last_year_prediction")
            _mark("log_same_transition_last_year_prediction")

        if {daily1, daily12, daily13, "days_in_month"}.issubset(combined.columns):
            combined["daily_same_transition_last_year_factor"] = _safe_divide_external(
                combined[daily12], combined[daily13], fill=np.nan
            )
            combined["daily_same_transition_last_year_factor_clipped"] = (
                combined["daily_same_transition_last_year_factor"].clip(0.75, 1.35)
            )
            combined["daily_same_transition_last_year_prediction"] = (
                combined[daily1]
                * combined["daily_same_transition_last_year_factor_clipped"]
                * combined["days_in_month"]
            )
            combined["log_daily_same_transition_last_year_prediction"] = _safe_log_external(
                combined["daily_same_transition_last_year_prediction"], eps=eps
            )
            _mark("daily_same_transition_last_year_factor")
            _mark("daily_same_transition_last_year_factor_clipped")
            _mark("daily_same_transition_last_year_prediction")
            _mark("log_daily_same_transition_last_year_prediction")

            combined["global_daily_transition_factor"] = (
                combined.groupby(time_col, dropna=False)["daily_same_transition_last_year_factor_clipped"]
                .transform("median")
            )
            combined["global_transition_prediction"] = (
                combined[daily1] * combined["global_daily_transition_factor"] * combined["days_in_month"]
            )
            combined["log_global_transition_prediction"] = _safe_log_external(
                combined["global_transition_prediction"], eps=eps
            )
            _mark("global_daily_transition_factor")
            _mark("global_transition_prediction")
            _mark("log_global_transition_prediction")

            if group_cols_for_transition is None:
                group_cols_for_transition = []
                for col in ["region", "trading_area_category", "opening_date_category", "settlement"]:
                    if col in combined.columns:
                        group_cols_for_transition.append([col])
                if {"region", "trading_area_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "trading_area_category"])
                if {"region", "opening_date_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "opening_date_category"])

            for group_cols in group_cols_for_transition:
                existing = [c for c in group_cols if c in combined.columns]
                if not existing:
                    continue
                group_name = "__".join(existing)
                factor_col = f"{group_name}_daily_transition_factor"
                pred_col = f"{group_name}_transition_prediction"
                combined[factor_col] = (
                    combined.groupby([time_col] + existing, dropna=False)["daily_same_transition_last_year_factor_clipped"]
                    .transform("median")
                )
                combined[pred_col] = combined[daily1] * combined[factor_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(factor_col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

        # Blends
        blend_sources = {
            "blend_day_scaled_and_last_year_prediction": (
                "day_scaled_lag1_prediction",
                "daily_same_transition_last_year_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_global_prediction": (
                "day_scaled_lag1_prediction",
                "global_transition_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_recent_mean_prediction": (
                "day_scaled_lag1_prediction",
                "recent_daily_mean_3_prediction",
                0.60,
                0.40,
            ),
        }
        for out_col, (c1, c2, w1, w2) in blend_sources.items():
            if {c1, c2}.issubset(combined.columns):
                combined[out_col] = w1 * combined[c1] + w2 * combined[c2]
                combined[f"log_{out_col}"] = _safe_log_external(combined[out_col], eps=eps)
                _mark(out_col)
                _mark(f"log_{out_col}")

        ratio_pairs = {
            "day_scaled_to_lag1_ratio": ("day_scaled_lag1_prediction", f"{target_col}_lag_1"),
            "last_year_anchor_to_day_scaled_ratio": ("daily_same_transition_last_year_prediction", "day_scaled_lag1_prediction"),
            "global_anchor_to_day_scaled_ratio": ("global_transition_prediction", "day_scaled_lag1_prediction"),
            "recent_mean_anchor_to_day_scaled_ratio": ("recent_daily_mean_3_prediction", "day_scaled_lag1_prediction"),
        }
        for out_col, (a, b) in ratio_pairs.items():
            if {a, b}.issubset(combined.columns):
                combined[out_col] = _safe_divide_external(combined[a], combined[b], fill=np.nan)
                _mark(out_col)

        log_diff_pairs = [(1, 2), (1, 3), (1, 6), (1, 12), (2, 3), (3, 6), (12, 13)]
        for left, right in log_diff_pairs:
            a, b = f"{target_col}_lag_{left}", f"{target_col}_lag_{right}"
            if {a, b}.issubset(combined.columns):
                out_col = f"log_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[a], eps=eps) - _safe_log_external(combined[b], eps=eps)
                _mark(out_col)

            da, db = f"daily_{target_col}_lag_{left}", f"daily_{target_col}_lag_{right}"
            if {da, db}.issubset(combined.columns):
                out_col = f"log_daily_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[da], eps=eps) - _safe_log_external(combined[db], eps=eps)
                _mark(out_col)

    # ------------------------------------------------------------
    # 3. March 2024 growth coefficients
    # ------------------------------------------------------------
    if add_march_2024_features:
        march_2024_idx = (2024 - int(min_year)) * 12 + (3 - 1)
        after_march_2024 = combined[time_col].astype(int) > march_2024_idx

        hist = history_raw.copy()
        hist["calendar_month_num"] = _calendar_month_from_time(hist[time_col], min_year)
        hist["calendar_year"] = _year_from_time(hist[time_col], min_year)

        hist_2024 = hist[
            (hist["calendar_year"].astype(int) == 2024)
            & (hist["calendar_month_num"].isin([2, 3]))
            & hist[target_col].notna()
        ].copy()

        if not hist_2024.empty:
            store_pivot = (
                hist_2024.pivot_table(
                    index=id_col,
                    columns="calendar_month_num",
                    values=target_col,
                    aggfunc="first",
                )
                .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                .reset_index()
            )
            if {"pto_feb_2024", "pto_mar_2024"}.issubset(store_pivot.columns):
                store_pivot["store_feb_mar_growth_2024"] = _safe_divide_external(
                    store_pivot["pto_mar_2024"], store_pivot["pto_feb_2024"], fill=np.nan
                ).clip(clip_growth_low, clip_growth_high)
                combined = combined.drop(columns=["store_feb_mar_growth_2024"], errors="ignore")
                combined = combined.merge(
                    store_pivot[[id_col, "store_feb_mar_growth_2024"]],
                    on=id_col,
                    how="left",
                )
                combined.loc[~after_march_2024, "store_feb_mar_growth_2024"] = np.nan
                _mark("store_feb_mar_growth_2024")

            def _add_group_growth(group_col, out_col):
                nonlocal combined
                if group_col not in hist_2024.columns or group_col not in combined.columns:
                    return
                group_agg = (
                    hist_2024.groupby([group_col, "calendar_month_num"], dropna=False)[target_col]
                    .sum()
                    .reset_index()
                )
                pivot = (
                    group_agg.pivot_table(
                        index=group_col,
                        columns="calendar_month_num",
                        values=target_col,
                        aggfunc="first",
                    )
                    .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                    .reset_index()
                )
                if {"pto_feb_2024", "pto_mar_2024"}.issubset(pivot.columns):
                    pivot[out_col] = _safe_divide_external(
                        pivot["pto_mar_2024"], pivot["pto_feb_2024"], fill=np.nan
                    ).clip(clip_growth_low, clip_growth_high)
                    combined = combined.drop(columns=[out_col], errors="ignore")
                    combined = combined.merge(pivot[[group_col, out_col]], on=group_col, how="left")
                    combined.loc[~after_march_2024, out_col] = np.nan
                    _mark(out_col)

            _add_group_growth("region", "region_feb_mar_growth_2024")
            _add_group_growth("settlement", "settlement_feb_mar_growth_2024")
            _add_group_growth("trading_area_category", "trading_area_feb_mar_growth_2024")

            feb_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 2, target_col].sum()
            mar_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 3, target_col].sum()
            global_growth = np.nan
            if pd.notna(feb_sum) and feb_sum > 0:
                global_growth = np.clip(mar_sum / feb_sum, clip_growth_low, clip_growth_high)

            combined["global_feb_mar_growth_2024"] = np.where(after_march_2024, global_growth, np.nan)
            _mark("global_feb_mar_growth_2024")

            growth_cols = [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_category_feb_mar_growth_2024",  # backward-safe typo guard
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]
            # unify expected column name from the prompt
            if "trading_area_category_feb_mar_growth_2024" in combined.columns and "trading_area_feb_mar_growth_2024" not in combined.columns:
                combined["trading_area_feb_mar_growth_2024"] = combined["trading_area_category_feb_mar_growth_2024"]
                _mark("trading_area_feb_mar_growth_2024")

            for col in [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]:
                if col not in combined.columns or f"{target_col}_lag_1" not in combined.columns:
                    continue
                combined[col] = combined[col].fillna(combined["global_feb_mar_growth_2024"])
                pred_col = f"lag1_x_{col}"
                combined[pred_col] = combined[f"{target_col}_lag_1"] * combined[col]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            if {"day_scaled_lag1_prediction", "lag1_x_store_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["blend_day_scaled_and_store_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_store_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_store_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_store_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_store_march_growth")
                _mark("log_blend_day_scaled_and_store_march_growth")

            if {"day_scaled_lag1_prediction", "lag1_x_region_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["blend_day_scaled_and_region_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_region_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_region_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_region_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_region_march_growth")
                _mark("log_blend_day_scaled_and_region_march_growth")

    # ------------------------------------------------------------
    # 4. Hierarchical share features
    # ------------------------------------------------------------
    if add_hierarchy_share_features:
        pto_lag_1 = f"{target_col}_lag_1"
        if pto_lag_1 in combined.columns:
            if "region" in combined.columns:
                combined["region_pto_lag1_mean"] = (
                    combined.groupby([time_col, "region"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_region_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["region_pto_lag1_mean"], fill=np.nan
                )
                _mark("region_pto_lag1_mean")
                _mark("store_share_region_lag1")

            if "settlement" in combined.columns:
                combined["settlement_pto_lag1_mean"] = (
                    combined.groupby([time_col, "settlement"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_settlement_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["settlement_pto_lag1_mean"], fill=np.nan
                )
                _mark("settlement_pto_lag1_mean")
                _mark("store_share_settlement_lag1")

            if "trading_area_category" in combined.columns:
                combined["area_category_pto_lag1_mean"] = (
                    combined.groupby([time_col, "trading_area_category"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_area_category_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["area_category_pto_lag1_mean"], fill=np.nan
                )
                _mark("area_category_pto_lag1_mean")
                _mark("store_share_area_category_lag1")

            if "region" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"region_pto_lag{lag}_mean"
                    share_col = f"store_share_region_lag{lag}"
                    combined[mean_col] = (
                        combined.groupby([time_col, "region"], dropna=False)[lag_col].transform("mean")
                    )
                    combined[share_col] = _safe_divide_external(combined[lag_col], combined[mean_col], fill=np.nan)
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [c for c in ["store_share_region_lag1", "store_share_region_lag2", "store_share_region_lag3"] if c in combined.columns]
                if share_cols:
                    combined["store_share_region_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_region_mean_3")
                if {"store_share_region_lag1", "store_share_region_lag2"}.issubset(combined.columns):
                    combined["store_share_region_change_1_2"] = _safe_divide_external(
                        combined["store_share_region_lag1"], combined["store_share_region_lag2"], fill=np.nan
                    )
                    combined["store_share_region_change_1_2_clipped"] = (
                        combined["store_share_region_change_1_2"].clip(0.70, 1.35)
                    )
                    _mark("store_share_region_change_1_2")
                    _mark("store_share_region_change_1_2_clipped")

            if "settlement" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"settlement_pto_lag{lag}_mean"
                    share_col = f"store_share_settlement_lag{lag}"
                    combined[mean_col] = (
                        combined.groupby([time_col, "settlement"], dropna=False)[lag_col].transform("mean")
                    )
                    combined[share_col] = _safe_divide_external(combined[lag_col], combined[mean_col], fill=np.nan)
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [
                    c for c in ["store_share_settlement_lag1", "store_share_settlement_lag2", "store_share_settlement_lag3"]
                    if c in combined.columns
                ]
                if share_cols:
                    combined["store_share_settlement_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_settlement_mean_3")

            if {"store_share_region_mean_3", "lag1_x_region_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["region_anchor_x_store_share_mean_3"] = (
                    combined["lag1_x_region_feb_mar_growth_2024"]
                    * combined["store_share_region_mean_3"].clip(0.50, 1.80)
                )
                combined["log_region_anchor_x_store_share_mean_3"] = _safe_log_external(
                    combined["region_anchor_x_store_share_mean_3"], eps=eps
                )
                _mark("region_anchor_x_store_share_mean_3")
                _mark("log_region_anchor_x_store_share_mean_3")

            if {"store_share_region_change_1_2_clipped", "day_scaled_lag1_prediction"}.issubset(combined.columns):
                combined["day_scaled_x_region_share_change"] = (
                    combined["day_scaled_lag1_prediction"]
                    * combined["store_share_region_change_1_2_clipped"]
                )
                combined["log_day_scaled_x_region_share_change"] = _safe_log_external(
                    combined["day_scaled_x_region_share_change"], eps=eps
                )
                _mark("day_scaled_x_region_share_change")
                _mark("log_day_scaled_x_region_share_change")

    # ------------------------------------------------------------
    # 5. Operational lag features
    # ------------------------------------------------------------
    if add_operational_lag_features:
        # lag_1, lag_3, lag_12 из исходных operational cols
        for col in lagged_monthly_cols:
            if col not in history_raw.columns:
                continue
            for lag in [1, 3, 12]:
                _merge_history_lag(history_raw, col, lag, f"{col}_lag_{lag}", [id_col])

            # rolling mean/median по прошлым 3 наблюдениям.
            raw_combined = pd.concat(
                [
                    history_raw.assign(__part_roll__="history"),
                    future_raw.assign(__part_roll__="future"),
                ],
                axis=0,
                ignore_index=True,
                sort=False,
            )
            raw_combined = _ensure_time_year_month_external(
                raw_combined,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                min_year=min_year,
                name=f"raw_combined_for_{col}",
            )
            raw_combined = raw_combined.sort_values([id_col, time_col], kind="mergesort")
            raw_combined["__x__"] = np.where(raw_combined["__part_roll__"].eq("history"), raw_combined[col], np.nan)
            shifted = raw_combined.groupby(id_col, sort=False)["__x__"].shift(1)
            raw_combined[f"{col}_roll_mean_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            raw_combined[f"{col}_roll_median_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .median()
                .reset_index(level=0, drop=True)
            )
            roll_map = raw_combined[[id_col, time_col, f"{col}_roll_mean_3", f"{col}_roll_median_3"]].drop_duplicates(
                [id_col, time_col],
                keep="last",
            )
            combined = combined.drop(columns=[f"{col}_roll_mean_3", f"{col}_roll_median_3"], errors="ignore")
            combined = combined.merge(roll_map, on=[id_col, time_col], how="left")
            _mark(f"{col}_roll_mean_3")
            _mark(f"{col}_roll_median_3")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1"}.issubset(combined.columns):
            combined["pto_per_working_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], combined["working_hours_per_day_lag_1"], fill=np.nan
            )
            combined["log_pto_per_working_hour_lag1"] = _safe_log_external(
                combined["pto_per_working_hour_lag1"], eps=eps
            )
            _mark("pto_per_working_hour_lag1")
            _mark("log_pto_per_working_hour_lag1")

        if {f"{target_col}_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            combined["pto_per_cash_register_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"],
                combined["number_of_cash_registers"].astype(float),
                fill=np.nan,
            )
            combined["log_pto_per_cash_register_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_lag1")
            _mark("log_pto_per_cash_register_lag1")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            denom = combined["working_hours_per_day_lag_1"] * combined["number_of_cash_registers"].astype(float)
            combined["pto_per_cash_register_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], denom, fill=np.nan
            )
            combined["log_pto_per_cash_register_hour_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_hour_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_hour_lag1")
            _mark("log_pto_per_cash_register_hour_lag1")

    # ------------------------------------------------------------
    # Final cleanup and restore parts
    # ------------------------------------------------------------
    created = sorted(set(c for c in created if c in combined.columns))
    combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)

    if fill_history_na and created:
        combined[created] = combined[created].fillna(fill_value)

    combined = combined.drop(columns=["calendar_year"], errors="ignore")

    history_out = (
        combined[combined["__part__"] == "history"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )
    future_out = (
        combined[combined["__part__"] == "future"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )

    # Align columns
    ordered_cols = list(history_out.columns)
    for c in future_out.columns:
        if c not in ordered_cols:
            ordered_cols.append(c)
    for c in ordered_cols:
        if c not in history_out.columns:
            history_out[c] = np.nan
        if c not in future_out.columns:
            future_out[c] = np.nan
    history_out = history_out[ordered_cols]
    future_out = future_out[ordered_cols]

    if downcast_float_features:
        for df_ in (history_out, future_out):
            float_cols = [c for c in df_.select_dtypes(include=["float64"]).columns if c != target_col]
            if float_cols:
                df_[float_cols] = df_[float_cols].astype("float32")

    return history_out, future_out, created


def build_retail_features_with_split(
    df: pd.DataFrame,
    *,
    # split config
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    create_test_if_missing: bool = True,
    expected_test_size: Optional[int] = 18657,
    future_unknown_cols: Optional[Sequence[str]] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    # feature config
    use_val_as_test_history: bool = True,
    add_integrated_anchor_features: bool = True,
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    # pass-through to base generator
    base_feature_kwargs: dict | None = None,
    verbose: bool = True,
) -> dict:
    """
    Единая функция:
    1) делает leakage-safe split;
    2) создаёт artificial test month, если нужно;
    3) запускает базовый generate_retail_forecast_features_leakfree;
    4) добавляет все новые идеи:
       - month_day_ratio_to_previous_month;
       - day_scaled_lag1_prediction;
       - expanding calendar heuristic factors/predictions;
       - daily_pto_lag_*;
       - recent daily baselines;
       - trend-corrected anchors;
       - same-transition-last-year anchors;
       - global/group transition anchors;
       - March 2024 growth coefficients;
       - hierarchical share features;
       - lagged operational features.
    """
    base_feature_kwargs = {} if base_feature_kwargs is None else dict(base_feature_kwargs)
    # Defaults chosen to cover the full feature-idea corpus. User can override them in base_feature_kwargs.
    base_feature_kwargs.setdefault("target_rolling_windows", (3, 6, 12))
    base_feature_kwargs.setdefault("monthly_cross_lags", (1, 3, 12))
    base_feature_kwargs.setdefault(
        "monthly_cross_feature_types",
        ("promo_share", "promo_x_items", "items_x_hours", "cancellations_per_item", "cancellations_per_hour"),
    )

    split_result = make_panel_time_split(
        df,
        group_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        split_mode=split_mode,
        val_time_idx=val_time_idx,
        val_year=val_year,
        val_month=val_month,
        test_time_idx=test_time_idx,
        test_year=test_year,
        test_month=test_month,
        min_year=min_year,
        create_test_if_missing=create_test_if_missing,
        future_unknown_cols=future_unknown_cols,
        expected_test_size=expected_test_size if split_mode in {"train_val_test", "train_test"} else None,
        verbose=verbose,
    )

    if split_mode == "train_val":
        train_df, val_df = split_result
        test_df = None
    elif split_mode == "train_val_test":
        train_df, val_df, test_df = split_result
    else:
        train_df, test_df = split_result
        val_df = None

    output = generate_retail_forecast_features_leakfree(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        use_val_as_test_history=use_val_as_test_history,
        verbose=verbose,
        **base_feature_kwargs,
    )

    output["raw_splits"] = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    if add_integrated_anchor_features:
        # validation pair: history = train, future = val
        if output.get("train_features") is not None and output.get("val_features") is not None:
            train_feat, val_feat, created = _add_integrated_anchor_features_pair(
                output["train_features"],
                output["val_features"],
                train_df,
                val_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                train_feat = train_feat.drop(columns=[year_col], errors="ignore")
                val_feat = val_feat.drop(columns=[year_col], errors="ignore")
            output["train_features"] = train_feat
            output["val_features"] = val_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})["val_created"] = created
            if verbose:
                print("Added integrated anchor features to train/val:", len(created))

        # test pair: history = train + val if requested and val exists, otherwise train
        if output.get("test_train_features") is not None and output.get("test_features") is not None:
            if val_df is not None and use_val_as_test_history:
                test_history_raw = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
            else:
                test_history_raw = train_df.copy()

            test_train_feat, test_feat, created = _add_integrated_anchor_features_pair(
                output["test_train_features"],
                output["test_features"],
                test_history_raw,
                test_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                test_train_feat = test_train_feat.drop(columns=[year_col], errors="ignore")
                test_feat = test_feat.drop(columns=[year_col], errors="ignore")
            output["test_train_features"] = test_train_feat
            output["test_features"] = test_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})["test_created"] = created
            if verbose:
                print("Added integrated anchor features to train/test:", len(created))

    # Recompute model columns after enhancement.
    ref = output["test_train_features"] if output.get("test_train_features") is not None else output.get("train_features")
    if ref is not None:
        exclude = {target_col}
        output["feature_cols"] = [c for c in ref.columns if c not in exclude]
        output["categorical_cols"] = [
            c for c in output["feature_cols"]
            if c in ref.columns and (
                pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c])
            )
        ]

    gc.collect()
    return output


# ============================================================
# Target engineering utilities
# ============================================================

def make_forecast_target(
    df: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    eps: float = 1e-9,
) -> pd.Series:
    """
    Builds model target for the main target formulations used in the experiments.

    Required feature columns by mode:
    - growth_log: pto_lag_1;
    - day_scaled_residual_log: day_scaled_lag1_prediction;
    - calendar_residual_log: calendar_heuristic_prediction;
    - daily_rate_log: n_days_in_month or days_in_month;
    - direct_log1p: only target_col.
    """
    if target_col not in df.columns:
        raise ValueError(f"В df нет target_col='{target_col}'.")

    y = pd.to_numeric(df[target_col], errors="coerce").astype(float)

    if mode == "original":
        return y

    if mode == "direct_log1p":
        return np.log1p(y.clip(lower=0))

    if mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in df.columns else "days_in_month"
        if day_col not in df.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(df[day_col], errors="coerce").astype(float)
        daily_rate = y / days.replace(0, np.nan)
        return np.log(np.clip(daily_rate, eps, None))

    raise ValueError(f"Неизвестный mode='{mode}'.")


def inverse_forecast_target(
    raw_prediction,
    features: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    residual_shrinkage: float = 1.0,
    clip_min: float = 0.0,
) -> np.ndarray:
    """
    Converts model output back to RTO forecast.

    residual_shrinkage is useful for residual modes, e.g. 0.35 or 0.50.
    """
    pred = np.asarray(raw_prediction, dtype=float)

    if mode == "original":
        out = pred
    elif mode == "direct_log1p":
        out = np.expm1(pred)
    elif mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(pred)
    elif mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in features.columns else "days_in_month"
        if day_col not in features.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(features[day_col], errors="coerce").astype(float).to_numpy()
        out = days * np.exp(pred)
    else:
        raise ValueError(f"Неизвестный mode='{mode}'.")

    out = np.asarray(out, dtype=float)
    out = np.where(np.isfinite(out), out, np.nan)
    if clip_min is not None:
        out = np.maximum(out, clip_min)
    return out

## Финальный CatBoost-пайплайн

In [25]:
# -*- coding: utf-8 -*-
"""
Synchronized final training / February-holdout pipeline for the current X5/RTO notebook.

This module is intended to be used together with the adaptive ranked-add feature
selection v4 result (`result_ranked_add`). It fixes the main inconsistency of the
old training function: selected features may contain categorical `*_cat` copies
created by the selector, so the final train/valid/test frames must recreate the
same categorical policy before resolving features and cat_features.

Main entry point:
    run_feb_holdout_selected_split_pipeline_v2(...)

Typical usage in notebook:
    result_final = run_feb_holdout_selected_split_pipeline_v2(
        df=df,
        feature_builder_func=build_retail_features_with_split,
        feature_selection_result=result_ranked_add,
        feature_builder_kwargs=make_x5_training_builder_kwargs(),
    )
"""

from __future__ import annotations

import copy
import os
import warnings
from typing import Any, Callable, Optional, Sequence

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import inv_boxcox
from catboost import CatBoostRegressor, Pool


# =============================================================================
# Shared X5 defaults
# =============================================================================


def _unique_keep_order(items: Sequence[Any] | None) -> list:
    """Return unique values preserving order."""
    if items is None:
        return []
    seen = set()
    out = []
    for x in items:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def make_x5_retail_categorical_source_cols(
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
) -> list[str]:
    """
    Exact categorical source columns requested for the current X5/RTO task.

    Object/string columns are used directly as CatBoost categorical features.
    Numeric/discrete columns are exposed via '<col>_cat' copies while preserving
    their original numeric representation.
    """
    return _unique_keep_order([
        id_col,
        "opening_date_category",
        "trading_area_category",
        "settlement",
        "region",
        "alcohol_license_flag",
        year_col,
        month_col,
        "marketplaces_deliveries_parcel_lockers_100m",
        "medical_institutions_and_pharmacies_300m",
        "schools_300m",
        "public_transport_stops_300m",
        "grocery_stores_500m",
        "pyaterochka_stores_500m",
        "number_of_cash_registers",
        "working_hours_per_day",
    ])


def make_x5_string_categorical_features() -> list[str]:
    """String/object categorical columns from the original renamed dataset."""
    return [
        "opening_date_category",
        "trading_area_category",
        "settlement",
        "region",
    ]


def make_x5_training_builder_kwargs(*, keep_year_col: bool = False) -> dict:
    """
    Feature-builder kwargs synchronized with feature selection v4 defaults.

    keep_year_col=False matches the selection defaults. Set keep_year_col=True if
    you intentionally want `year` / `year_cat` to be available and you also used
    the same option during feature selection.
    """
    return {
        "base_feature_kwargs": {
            "target_history_scale": "log",
            "monthly_history_scale": "raw",
            "target_rolling_windows": (3, 6, 12),
            "monthly_cross_lags": (1, 3, 12),
            "monthly_cross_feature_types": (
                "promo_share",
                "promo_x_items",
                "items_x_hours",
                "cancellations_per_item",
                "cancellations_per_hour",
            ),
            "drop_current_monthly_cols": True,
            "drop_current_monthly_derived_cols": True,
            "drop_year_col": not bool(keep_year_col),
            "drop_obvious_duplicates": False,
            "fill_history_na": False,
            "downcast_float_features": True,
        }
    }


# =============================================================================
# Metrics / target adapter
# =============================================================================


def _mape_percent(y_true, y_pred, eps: float = 1e-9) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (np.abs(y_true) > eps)
    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask])) * 100.0)


def _competition_score_from_mape(mape: float) -> float:
    if pd.isna(mape):
        return np.nan
    return float(100.0 * ((100.0 - min(float(mape), 100.0)) / 100.0) ** 2)


class _TargetAdapter:
    """Target transformer/inverter for final training specs."""

    def __init__(
        self,
        mode: str,
        *,
        target_col: str = "pto",
        lag_col: str = "pto_lag_1",
        eps: float = 1e-9,
    ):
        self.mode = str(mode)
        self.target_col = target_col
        self.lag_col = lag_col
        self.eps = eps
        self.lambda_: Optional[float] = None
        self.offset_: float = 0.0

    def fit(self, train_df: pd.DataFrame) -> "_TargetAdapter":
        y = train_df[self.target_col].astype(float).values

        if self.mode == "boxcox_level":
            min_y = np.nanmin(y)
            self.offset_ = 0.0 if min_y > 0 else -min_y + self.eps

            y_pos = y + self.offset_
            if np.any(y_pos <= 0):
                raise ValueError("После offset в Box-Cox остались неположительные значения.")

            self.lambda_ = float(stats.boxcox_normmax(y_pos, method="mle"))

        return self

    def valid_mask(self, frame: pd.DataFrame, *, require_target: bool = True) -> pd.Series:
        if require_target:
            if self.target_col not in frame.columns:
                raise ValueError(f"В frame нет target_col='{self.target_col}'.")
            y = frame[self.target_col]
            mask = y.notna()
        else:
            mask = pd.Series(True, index=frame.index)

        if self.mode in ["log_level", "boxcox_level"]:
            if require_target:
                mask &= frame[self.target_col].astype(float) > 0

        elif self.mode == "log_diff_1":
            if self.lag_col not in frame.columns:
                raise ValueError(
                    f"Для mode='log_diff_1' нужен лаговый столбец '{self.lag_col}'."
                )

            lag = frame[self.lag_col]
            mask &= lag.notna()
            mask &= lag.astype(float) > 0

            if require_target:
                mask &= frame[self.target_col].astype(float) > 0

        elif self.mode == "original":
            pass

        else:
            raise ValueError(f"Неизвестный target mode: {self.mode}")

        return mask

    def transform_y(self, frame: pd.DataFrame) -> np.ndarray:
        y = frame[self.target_col].astype(float).values

        if self.mode == "original":
            return y

        if self.mode == "log_level":
            return np.log(np.maximum(y, self.eps))

        if self.mode == "log_diff_1":
            lag = frame[self.lag_col].astype(float).values
            return np.log(np.maximum(y, self.eps) / np.maximum(lag, self.eps))

        if self.mode == "boxcox_level":
            if self.lambda_ is None:
                raise ValueError("Для Box-Cox сначала нужно вызвать fit().")
            y_pos = y + self.offset_
            return stats.boxcox(y_pos, lmbda=self.lambda_)

        raise ValueError(f"Неизвестный target mode: {self.mode}")

    def _safe_inv_boxcox(self, z) -> np.ndarray:
        z = np.asarray(z, dtype=float)

        if self.lambda_ is None:
            raise ValueError("lambda_ не задана.")

        if np.isclose(self.lambda_, 0.0):
            return np.exp(z)

        if self.lambda_ > 0:
            lower_bound = -1.0 / self.lambda_ + self.eps
            z = np.maximum(z, lower_bound)

        if self.lambda_ < 0:
            upper_bound = -1.0 / self.lambda_ - self.eps
            z = np.minimum(z, upper_bound)

        return inv_boxcox(z, self.lambda_)

    def inverse_pred_to_level(self, pred_transformed, frame: pd.DataFrame) -> np.ndarray:
        pred_transformed = np.asarray(pred_transformed, dtype=float)

        if self.mode == "original":
            pred = pred_transformed

        elif self.mode == "log_level":
            pred = np.exp(pred_transformed)

        elif self.mode == "log_diff_1":
            if self.lag_col not in frame.columns:
                raise ValueError(f"Для inverse log_diff_1 нужен lag_col='{self.lag_col}'.")
            lag = frame[self.lag_col].astype(float).values
            pred = lag * np.exp(pred_transformed)

        elif self.mode == "boxcox_level":
            pred = self._safe_inv_boxcox(pred_transformed) - self.offset_

        else:
            raise ValueError(f"Неизвестный target mode: {self.mode}")

        pred = np.asarray(pred, dtype=float)
        pred = np.where(np.isfinite(pred), pred, np.nan)
        pred = np.maximum(pred, 0.0)

        return pred


# =============================================================================
# Categorical policy for final training frames
# =============================================================================


def _is_object_like(s: pd.Series) -> bool:
    return (
        pd.api.types.is_object_dtype(s)
        or pd.api.types.is_string_dtype(s)
        or isinstance(s.dtype, pd.CategoricalDtype)
    )


def _prepare_series_as_cat(s: pd.Series, missing_token: str) -> pd.Series:
    out = s.astype("object")
    out = out.where(pd.notna(out), missing_token)
    return out.astype(str)


def _cat_copy_name(col: str, suffix: str = "_cat") -> str:
    return f"{col}{suffix}"


def _apply_x5_categorical_policy_to_frames(
    frames: dict[str, pd.DataFrame],
    *,
    categorical_source_cols: Sequence[str],
    explicit_string_cat_features: Sequence[str],
    target_col: str = "pto",
    categorical_copy_suffix: str = "_cat",
    create_numeric_categorical_copies: bool = True,
    categorical_missing_token: str = "__MISSING__",
    verbose: bool = True,
) -> tuple[dict[str, pd.DataFrame], list[str], pd.DataFrame]:
    """
    Apply the same categorical policy as feature selection v4 to arbitrary frames.

    - object/string/category source columns are converted in-place and used as cat_features;
    - numeric/discrete source columns are duplicated as '<col>_cat';
    - original numeric columns are preserved.
    """
    source_cols = _unique_keep_order(categorical_source_cols)
    explicit_strings = _unique_keep_order(explicit_string_cat_features)

    out_frames: dict[str, pd.DataFrame] = {}
    cat_features: list[str] = []
    records: list[dict[str, Any]] = []

    for frame_name, frame in frames.items():
        df_part = frame.copy()

        for src_col in source_cols:
            if src_col not in df_part.columns or src_col == target_col:
                continue

            if _is_object_like(df_part[src_col]):
                cat_col = src_col
                df_part[cat_col] = _prepare_series_as_cat(df_part[src_col], categorical_missing_token)
                representation = "original_object"
            else:
                if create_numeric_categorical_copies:
                    cat_col = _cat_copy_name(src_col, categorical_copy_suffix)
                    df_part[cat_col] = _prepare_series_as_cat(df_part[src_col], categorical_missing_token)
                    representation = "numeric_copy"
                else:
                    cat_col = src_col
                    df_part[cat_col] = _prepare_series_as_cat(df_part[src_col], categorical_missing_token)
                    representation = "numeric_overwritten_as_string"

            cat_features.append(cat_col)
            records.append({
                "frame": frame_name,
                "source_col": src_col,
                "cat_feature": cat_col,
                "representation": representation,
                "n_unique": int(df_part[cat_col].nunique(dropna=False)),
            })

        for cat_col in explicit_strings:
            if cat_col in df_part.columns and cat_col != target_col:
                df_part[cat_col] = _prepare_series_as_cat(df_part[cat_col], categorical_missing_token)
                cat_features.append(cat_col)

        out_frames[frame_name] = df_part

    cat_features = _unique_keep_order(cat_features)
    meta = pd.DataFrame(records)
    if not meta.empty:
        meta = meta.drop_duplicates(
            subset=["frame", "source_col", "cat_feature"],
            keep="last",
        )

    if verbose:
        print("\n[categorical-policy-final]")
        print(f"cat_features created/available: {len(cat_features)}")
        if not meta.empty:
            print(
                meta[["source_col", "cat_feature", "representation"]]
                .drop_duplicates()
                .to_string(index=False)
            )
        print()

    return out_frames, cat_features, meta


# =============================================================================
# Default specs
# =============================================================================


def make_default_candidate_specs_gpu(
    *,
    verbose: bool = True,
    devices: str = "0",
    random_seed: int = 42,
    iterations: int = 3000,
    learning_rate: float = 0.01,
    depth: int = 6,
    early_stopping_rounds: int = 50,
    metric_period: int = 50,
) -> list[dict]:
    """Default GPU specs: original / log_level / log_diff_1 / boxcox_level."""
    base_model_params_gpu = {
        "iterations": int(iterations),
        "learning_rate": float(learning_rate),
        "depth": int(depth),
        "loss_function": "MAE",
        "eval_metric": "MAE",
        "random_seed": int(random_seed),
        "allow_writing_files": False,
        "task_type": "GPU",
        "devices": devices,
        "metric_period": int(metric_period),
        "verbose": False,
    }

    base_fit_params = {
        "early_stopping_rounds": int(early_stopping_rounds),
        "verbose": 50 if verbose else False,
    }

    return [
        {
            "name": "original_gpu",
            "target_mode": "original",
            "model_params": copy.deepcopy(base_model_params_gpu),
            "fit_params": copy.deepcopy(base_fit_params),
        },
        {
            "name": "log_level_gpu",
            "target_mode": "log_level",
            "model_params": copy.deepcopy(base_model_params_gpu),
            "fit_params": copy.deepcopy(base_fit_params),
        },
        {
            "name": "log_diff_1_gpu",
            "target_mode": "log_diff_1",
            "model_params": copy.deepcopy(base_model_params_gpu),
            "fit_params": copy.deepcopy(base_fit_params),
        },
        {
            "name": "boxcox_level_gpu",
            "target_mode": "boxcox_level",
            "model_params": copy.deepcopy(base_model_params_gpu),
            "fit_params": copy.deepcopy(base_fit_params),
        },
    ]


# =============================================================================
# Main final training pipeline
# =============================================================================


def run_feb_holdout_selected_split_pipeline_v2(
    df: pd.DataFrame,
    feature_builder_func: Callable,
    feature_builder_kwargs: Optional[dict] = None,

    # features
    # features
    features_to_use: Optional[Sequence[str]] = None,      # простой ручной список фичей
    feature_selection_result: Optional[dict] = None,     # результат feature selection, опционально
    selected_features: Optional[Sequence[str]] = None,   # старый алиас, можно оставить для совместимости
    explicit_features: Optional[Sequence[str]] = None,   # старый алиас, можно оставить для совместимости
    explicit_cat_features: Optional[Sequence[str]] = None,
    default_cat_candidates: Optional[Sequence[str]] = None,
    use_pruned_features_if_available: bool = True,

    # categorical policy synced with selector v4
    use_x5_categorical_policy: bool = True,
    categorical_source_cols: Optional[Sequence[str]] = None,
    explicit_string_cat_features: Optional[Sequence[str]] = None,
    create_numeric_categorical_copies: bool = True,
    categorical_copy_suffix: str = "_cat",
    categorical_missing_token: str = "__MISSING__",
    use_cat_features_from_selection: bool = True,
    strict_selected_feature_check: bool = True,

    # specs
    candidate_specs: Optional[Sequence[dict]] = None,

    # dates
    forecast_year: int = 2025,
    forecast_month: int = 3,
    valid_year: int = 2025,
    valid_month: int = 2,

    # columns
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    lag_col: str = "pto_lag_1",
    min_year: int = 2023,

    # split / future rows
    create_test_if_missing: bool = True,
    expected_test_size: Optional[int] = None,
    future_unknown_cols: Sequence[str] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),

    # output
    save_submission_path: Optional[str] = "/content/90_13_outputs/test_feb_holdout_selected_gpu.csv",
    also_save_test_csv: bool = True,
    sort_submission_by_id: bool = True,
    cast_id_to_int: bool = True,

    # final refit
    min_final_iterations: int = 100,

    # logging
    verbose: bool = True,
    return_feature_outputs: bool = False,
) -> dict:
    """
    Synchronized February-holdout pipeline for selected X5 features.

    Key difference vs old `run_feb_holdout_selected_split_pipeline`:
    after building train/valid/final/test features, the function applies the same
    categorical policy as feature selection v4. This recreates `*_cat` features
    before selected features are resolved, so final training and feature selection
    work in the same feature space.
    """

    def _safe_call_feature_builder(
        split_mode: str,
        *,
        val_year: Optional[int] = None,
        val_month: Optional[int] = None,
        test_year: Optional[int] = None,
        test_month: Optional[int] = None,
    ) -> dict:
        kwargs = {} if feature_builder_kwargs is None else dict(feature_builder_kwargs)

        reserved = {
            "df",
            "split_mode",
            "id_col",
            "year_col",
            "month_col",
            "time_col",
            "target_col",
            "min_year",
            "val_year",
            "val_month",
            "test_year",
            "test_month",
            "create_test_if_missing",
            "expected_test_size",
            "future_unknown_cols",
            "verbose",
        }
        safe_kwargs = {k: v for k, v in kwargs.items() if k not in reserved}

        return feature_builder_func(
            df=df,
            split_mode=split_mode,
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
            val_year=val_year,
            val_month=val_month,
            test_year=test_year,
            test_month=test_month,
            create_test_if_missing=create_test_if_missing,
            expected_test_size=expected_test_size,
            future_unknown_cols=future_unknown_cols,
            verbose=verbose,
            **safe_kwargs,
        )

    def _get_required_frame(output: dict, possible_keys: Sequence[str], output_name: str) -> pd.DataFrame:
        for key in possible_keys:
            if key in output and isinstance(output[key], pd.DataFrame):
                return output[key]
        raise KeyError(
            f"В output для {output_name} не найден ни один из ключей: {possible_keys}. "
            f"Доступные ключи: {list(output.keys())}"
        )

    def _infer_features_from_output(output: dict, ref_frame: pd.DataFrame) -> list[str]:
        if "feature_cols" in output and output["feature_cols"] is not None:
            feats = list(output["feature_cols"])
        else:
            exclude = {target_col}
            feats = [c for c in ref_frame.columns if c not in exclude]
        feats = [f for f in feats if f != target_col]
        return _unique_keep_order(feats)


    def _resolve_features(
        output: dict,
        train_frame: pd.DataFrame,
        val_frame: pd.DataFrame,
        final_train_frame: pd.DataFrame,
        final_test_frame: pd.DataFrame,
    ) -> list[str]:

        # Приоритет:
        # 1. features_to_use — основной новый вариант
        # 2. explicit_features — старый ручной вариант
        # 3. selected_features — старый алиас
        # 4. feature_selection_result — результат отбора
        # 5. все фичи из feature_builder output

        if features_to_use is not None:
            feats = list(features_to_use)

        elif explicit_features is not None:
            feats = list(explicit_features)

        elif selected_features is not None:
            feats = list(selected_features)

        elif feature_selection_result is not None:
            if (
                use_pruned_features_if_available
                and "selected_features_pruned" in feature_selection_result
            ):
                feats = list(feature_selection_result["selected_features_pruned"])
            elif "selected_features" in feature_selection_result:
                feats = list(feature_selection_result["selected_features"])
            else:
                raise KeyError(
                    "feature_selection_result должен содержать 'selected_features' "
                    "или 'selected_features_pruned'."
                )

        else:
            feats = _infer_features_from_output(output, train_frame)

        feats = _unique_keep_order(feats)

        if target_col in feats:
            raise ValueError(f"target_col='{target_col}' попал в features. Это leakage.")

        frames = {
            "train_features": train_frame,
            "val_features": val_frame,
            "final_train_features": final_train_frame,
            "test_features": final_test_frame,
        }
        missing_by_frame: dict[str, list[str]] = {}
        for name, frame in frames.items():
            missing = [f for f in feats if f not in frame.columns]
            if missing:
                missing_by_frame[name] = missing

        if missing_by_frame:
            msg = "Часть selected features отсутствует в построенных split-фреймах.\n"
            msg += "Проверь, что feature_builder_kwargs совпадают с feature selection, "
            msg += "и что categorical policy создала нужные *_cat признаки.\n"
            for name, missing in missing_by_frame.items():
                msg += f"\n{name}: {missing[:80]}"
                if len(missing) > 80:
                    msg += f" ... всего {len(missing)}"
            if strict_selected_feature_check:
                raise ValueError(msg)
            warnings.warn(msg)
            missing_all = set().union(*[set(v) for v in missing_by_frame.values()])
            feats = [f for f in feats if f not in missing_all]

        return feats

    def _resolve_cat_features(output: dict, features: Sequence[str], train_frame: pd.DataFrame) -> list[str]:
        feature_set = set(features)

        if explicit_cat_features is not None:
            cats = list(explicit_cat_features)
        else:
            cats = []

            if use_cat_features_from_selection and feature_selection_result is not None:
                cats += list(feature_selection_result.get("categorical_features_used", []) or [])
                cats += list(feature_selection_result.get("enhanced_categorical_features", []) or [])

            cats += list(output.get("categorical_cols", []) or [])

            if default_cat_candidates is None:
                # Exact X5 categories, no old automatic month_name/holiday/population defaults.
                base_sources = make_x5_retail_categorical_source_cols(
                    id_col=id_col,
                    year_col=year_col,
                    month_col=month_col,
                )
                base_strings = make_x5_string_categorical_features()
                base_numeric_copies = [_cat_copy_name(c, categorical_copy_suffix) for c in base_sources]
                cat_candidates = base_strings + base_numeric_copies
            else:
                cat_candidates = list(default_cat_candidates)

            cats += list(cat_candidates)

        # Additionally pick object/string/category selected features.
        for col in features:
            if col in train_frame.columns and _is_object_like(train_frame[col]):
                cats.append(col)

        cats = _unique_keep_order(cats)
        cats = [c for c in cats if c in feature_set and c != target_col]
        return cats

    def _prepare_catboost_frame(frame: pd.DataFrame, cat_features: Sequence[str]) -> pd.DataFrame:
        frame = frame.copy()
        for col in cat_features:
            if col in frame.columns:
                frame[col] = _prepare_series_as_cat(frame[col], categorical_missing_token)
        return frame

    def _spec_is_usable(
        spec: dict,
        train_frame: pd.DataFrame,
        val_frame: pd.DataFrame,
        final_train_frame: pd.DataFrame,
        final_test_frame: pd.DataFrame,
    ) -> bool:
        mode = spec["target_mode"]

        if mode == "log_diff_1":
            for frame_name, frame in {
                "train": train_frame,
                "val": val_frame,
                "final_train": final_train_frame,
                "test": final_test_frame,
            }.items():
                if lag_col not in frame.columns:
                    if verbose:
                        print(f"[skip spec] {spec['name']}: нет lag_col='{lag_col}' в {frame_name}.")
                    return False

        return True

    def _fit_eval_one_spec(
        spec: dict,
        train_frame: pd.DataFrame,
        val_frame: pd.DataFrame,
        features: Sequence[str],
        cat_features: Sequence[str],
    ) -> dict:
        mode = spec["target_mode"]
        params = copy.deepcopy(spec.get("model_params", {}))
        fit_params = copy.deepcopy(spec.get("fit_params", {}))

        adapter = _TargetAdapter(
            mode=mode,
            target_col=target_col,
            lag_col=lag_col,
        ).fit(train_frame)

        train_mask = adapter.valid_mask(train_frame, require_target=True)
        val_mask = adapter.valid_mask(val_frame, require_target=True)

        train_used = train_frame.loc[train_mask].copy()
        val_used = val_frame.loc[val_mask].copy()

        if train_used.empty:
            raise ValueError(f"Пустой train после valid_mask для spec={spec['name']}.")
        if val_used.empty:
            raise ValueError(f"Пустой valid после valid_mask для spec={spec['name']}.")

        y_train = adapter.transform_y(train_used)
        y_val_transformed = adapter.transform_y(val_used)

        train_used = _prepare_catboost_frame(train_used, cat_features)
        val_used = _prepare_catboost_frame(val_used, cat_features)

        train_pool = Pool(train_used[list(features)], y_train, cat_features=list(cat_features))
        val_pool = Pool(val_used[list(features)], y_val_transformed, cat_features=list(cat_features))

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, **fit_params)

        pred_transformed = model.predict(val_pool)
        pred_level = adapter.inverse_pred_to_level(pred_transformed, val_used)

        mape = _mape_percent(val_used[target_col].values, pred_level)
        score = _competition_score_from_mape(mape)

        best_iter = model.get_best_iteration()
        if best_iter is None:
            best_iter = -1

        return {
            "spec_name": spec["name"],
            "target_mode": mode,
            "fold_name": f"valid_{valid_year}_{valid_month:02d}",
            "valid_year": valid_year,
            "valid_month": valid_month,
            "fold_weight": 1.0,
            "mape": mape,
            "score": score,
            "n_train": len(train_used),
            "n_valid": len(val_used),
            "best_iteration": int(best_iter),
        }

    def _make_leaderboard(fold_results: pd.DataFrame) -> pd.DataFrame:
        leaderboard = (
            fold_results
            .assign(weighted_part=lambda x: x["mape"] * x["fold_weight"])
            .groupby(["spec_name", "target_mode"], as_index=False)
            .agg(
                weighted_mape=("weighted_part", "sum"),
                mean_mape=("mape", "mean"),
                std_mape=("mape", "std"),
                min_mape=("mape", "min"),
                max_mape=("mape", "max"),
                mean_score=("score", "mean"),
            )
            .sort_values("weighted_mape", ascending=True)
            .reset_index(drop=True)
        )
        leaderboard["weighted_score"] = leaderboard["weighted_mape"].apply(_competition_score_from_mape)
        return leaderboard

    def _fit_final_and_predict(
        best_spec: dict,
        final_train_frame: pd.DataFrame,
        final_test_frame: pd.DataFrame,
        features: Sequence[str],
        cat_features: Sequence[str],
        final_iterations: Optional[int] = None,
    ) -> tuple[CatBoostRegressor, _TargetAdapter, pd.DataFrame, dict]:
        spec = copy.deepcopy(best_spec)
        params = copy.deepcopy(spec.get("model_params", {}))

        if final_iterations is not None:
            params["iterations"] = int(final_iterations)

        mode = spec["target_mode"]
        adapter = _TargetAdapter(
            mode=mode,
            target_col=target_col,
            lag_col=lag_col,
        ).fit(final_train_frame)

        train_mask = adapter.valid_mask(final_train_frame, require_target=True)
        final_train_used = final_train_frame.loc[train_mask].copy()

        if final_train_used.empty:
            raise ValueError("Пустой final_train после valid_mask.")

        if mode == "log_diff_1":
            test_mask = adapter.valid_mask(final_test_frame, require_target=False)
            if not bool(test_mask.all()):
                bad_n = int((~test_mask).sum())
                raise ValueError(
                    f"В test есть строки с некорректным lag_col='{lag_col}' для log_diff_1: {bad_n}"
                )

        y_train = adapter.transform_y(final_train_used)

        final_train_used = _prepare_catboost_frame(final_train_used, cat_features)
        final_test_used = _prepare_catboost_frame(final_test_frame.copy(), cat_features)

        train_pool = Pool(final_train_used[list(features)], y_train, cat_features=list(cat_features))
        test_pool = Pool(final_test_used[list(features)], cat_features=list(cat_features))

        final_model = CatBoostRegressor(**params)
        # No eval_set at final refit; no early stopping here.
        final_model.fit(train_pool, verbose=False)

        pred_transformed = final_model.predict(test_pool)
        pred_level = adapter.inverse_pred_to_level(pred_transformed, final_test_used)

        submission = pd.DataFrame({
            id_col: final_test_used[id_col].values,
            "rto": pred_level,
        })

        return final_model, adapter, submission, params

    # -------------------------------------------------------------------------
    # 1. Build validation split
    # -------------------------------------------------------------------------

    if feature_builder_kwargs is None:
        feature_builder_kwargs = make_x5_training_builder_kwargs()
    else:
        feature_builder_kwargs = dict(feature_builder_kwargs)

    if candidate_specs is None:
        candidate_specs = make_default_candidate_specs_gpu(verbose=verbose)
    else:
        candidate_specs = copy.deepcopy(list(candidate_specs))

    if categorical_source_cols is None:
        categorical_source_cols = make_x5_retail_categorical_source_cols(
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
        )
    else:
        categorical_source_cols = list(categorical_source_cols)

    if explicit_string_cat_features is None:
        explicit_string_cat_features = make_x5_string_categorical_features()
    else:
        explicit_string_cat_features = list(explicit_string_cat_features)

    if verbose:
        print("=" * 100)
        print("FEB HOLDOUT SELECTED SPLIT PIPELINE V2")
        print("=" * 100)
        print(f"valid:    {valid_year}-{valid_month:02d}")
        print(f"forecast: {forecast_year}-{forecast_month:02d}")
        print(f"builder:  {getattr(feature_builder_func, '__name__', str(feature_builder_func))}")
        print(f"x5 categorical policy: {use_x5_categorical_policy}")
        print("=" * 100)

    if verbose:
        print("\n[1/7] Build validation split features...")

    val_output = _safe_call_feature_builder(
        split_mode="train_val",
        val_year=valid_year,
        val_month=valid_month,
    )
    if not isinstance(val_output, dict):
        raise TypeError("feature_builder_func должен вернуть dict для split_mode='train_val'.")

    train_features = _get_required_frame(val_output, ["train_features"], "validation train")
    val_features = _get_required_frame(val_output, ["val_features", "valid_features"], "validation val")

    # -------------------------------------------------------------------------
    # 2. Build final train/test split
    # -------------------------------------------------------------------------

    if verbose:
        print("\n[2/7] Build final train/test features...")

    final_output = _safe_call_feature_builder(
        split_mode="train_test",
        test_year=forecast_year,
        test_month=forecast_month,
    )
    if not isinstance(final_output, dict):
        raise TypeError("feature_builder_func должен вернуть dict для split_mode='train_test'.")

    final_train_features = _get_required_frame(
        final_output,
        ["test_train_features", "train_features"],
        "final train",
    )
    test_features = _get_required_frame(final_output, ["test_features"], "final test")

    # -------------------------------------------------------------------------
    # 3. Apply categorical policy before resolving selected features
    # -------------------------------------------------------------------------

    categorical_policy_meta = pd.DataFrame()
    created_cat_features: list[str] = []

    if use_x5_categorical_policy:
        frames_out, created_cat_features, categorical_policy_meta = _apply_x5_categorical_policy_to_frames(
            {
                "train_features": train_features,
                "val_features": val_features,
                "final_train_features": final_train_features,
                "test_features": test_features,
            },
            categorical_source_cols=categorical_source_cols,
            explicit_string_cat_features=explicit_string_cat_features,
            target_col=target_col,
            categorical_copy_suffix=categorical_copy_suffix,
            create_numeric_categorical_copies=create_numeric_categorical_copies,
            categorical_missing_token=categorical_missing_token,
            verbose=verbose,
        )
        train_features = frames_out["train_features"]
        val_features = frames_out["val_features"]
        final_train_features = frames_out["final_train_features"]
        test_features = frames_out["test_features"]

    # -------------------------------------------------------------------------
    # 4. Resolve features and cat_features
    # -------------------------------------------------------------------------

    if verbose:
        print("\n[3/7] Resolve selected features and cat_features...")

    features = _resolve_features(
        val_output,
        train_features,
        val_features,
        final_train_features,
        test_features,
    )

    cat_features = _resolve_cat_features(val_output, features, train_features)

    if verbose:
        print(f"Features count: {len(features)}")
        print("Features:")
        for i, f in enumerate(features, start=1):
            print(f"{i:03d}. {f}")

        print(f"\nCat features count: {len(cat_features)}")
        print(cat_features)

    # -------------------------------------------------------------------------
    # 5. Filter specs
    # -------------------------------------------------------------------------

    usable_specs = []
    for spec in candidate_specs:
        if _spec_is_usable(
            spec,
            train_features,
            val_features,
            final_train_features,
            test_features,
        ):
            usable_specs.append(spec)

    if not usable_specs:
        raise ValueError("Не осталось usable candidate_specs.")

    if verbose:
        print("\nTarget specs:")
        for spec in usable_specs:
            print(f"- {spec['name']}: {spec['target_mode']}")

    # -------------------------------------------------------------------------
    # 6. Validation search
    # -------------------------------------------------------------------------

    if verbose:
        print("\n[4/7] Run feb holdout comparison...")

    rows = []
    for spec in usable_specs:
        if verbose:
            print("=" * 100)
            print(f"START SPEC: {spec['name']} | target_mode={spec['target_mode']}")
            print("=" * 100)

        row = _fit_eval_one_spec(spec, train_features, val_features, features, cat_features)
        rows.append(row)

        if verbose:
            print(row)

    fold_results = pd.DataFrame(rows)
    leaderboard = _make_leaderboard(fold_results)

    best_spec_name = leaderboard.loc[0, "spec_name"]
    best_spec = [s for s in usable_specs if s["name"] == best_spec_name][0]

    if verbose:
        print("\nLeaderboard:")
        print(leaderboard.to_string(index=False))
        print("\nBest spec:")
        print(best_spec["name"], "|", best_spec["target_mode"])

    # -------------------------------------------------------------------------
    # 7. Final refit / predict
    # -------------------------------------------------------------------------

    best_spec_for_final = copy.deepcopy(best_spec)
    best_iters = (
        fold_results
        .loc[fold_results["spec_name"] == best_spec["name"], "best_iteration"]
        .dropna()
    )
    best_iters = best_iters[best_iters >= 0]

    final_iterations = None
    if len(best_iters) > 0:
        final_iterations = int(np.median(best_iters)) + 1
        final_iterations = max(final_iterations, int(min_final_iterations))

        best_spec_for_final["model_params"] = copy.deepcopy(best_spec_for_final.get("model_params", {}))
        best_spec_for_final["model_params"]["iterations"] = final_iterations

    if verbose:
        print("\n[5/7] Final refit...")
        print("Final iterations:", final_iterations)

    final_model, final_adapter, submission, final_model_params = _fit_final_and_predict(
        best_spec_for_final,
        final_train_features,
        test_features,
        features,
        cat_features,
        final_iterations=final_iterations,
    )

    # -------------------------------------------------------------------------
    # 8. Submission normalization / save
    # -------------------------------------------------------------------------

    if cast_id_to_int:
        try:
            submission[id_col] = submission[id_col].astype(int)
        except Exception as e:
            raise ValueError(
                f"Не удалось привести submission['{id_col}'] к int. "
                "Если id строковый, поставь cast_id_to_int=False."
            ) from e

    if sort_submission_by_id:
        submission = submission.sort_values(id_col).reset_index(drop=True)

    if submission[id_col].duplicated().any():
        dup_n = int(submission[id_col].duplicated().sum())
        raise ValueError(f"В submission есть дубли по '{id_col}': {dup_n}")

    if save_submission_path is not None:
        save_dir = os.path.dirname(save_submission_path)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
        submission.to_csv(save_submission_path, index=False)

    if also_save_test_csv:
        os.makedirs("/content/90_13_outputs", exist_ok=True)
        submission.to_csv("/content/90_13_outputs/test.csv", index=False)

    if verbose:
        print("\n[6/7] Submission:")
        print(submission.shape)
        print(submission.head())

        if save_submission_path is not None:
            print("\nSaved submission:")
            print(save_submission_path)
            print(os.path.exists(save_submission_path))

        if also_save_test_csv:
            print("\nSaved /content/90_13_outputs/test.csv:")
            print(os.path.exists("/content/90_13_outputs/test.csv"))

    result = {
        "features": features,
        "cat_features": cat_features,
        "created_cat_features": created_cat_features,
        "categorical_policy_meta": categorical_policy_meta,
        "fold_results": fold_results,
        "leaderboard": leaderboard,
        "best_spec": best_spec,
        "best_spec_for_final": best_spec_for_final,
        "final_model_params": final_model_params,
        "final_model": final_model,
        "final_adapter": final_adapter,
        "submission": submission,
        "train_features_shape": train_features.shape,
        "val_features_shape": val_features.shape,
        "final_train_features_shape": final_train_features.shape,
        "test_features_shape": test_features.shape,
        "used_feature_selection_result": feature_selection_result is not None,
        "save_submission_path": save_submission_path,
    }

    if return_feature_outputs:
        result["val_output"] = val_output
        result["final_output"] = final_output
        result["train_features"] = train_features
        result["val_features"] = val_features
        result["final_train_features"] = final_train_features
        result["test_features"] = test_features

    return result

In [26]:
def make_candidate_specs_for_task(
    *,
    task_type: str,
    devices: str = "0",
    verbose: bool = True,
    random_seed: int = RANDOM_STATE,
    target_modes: tuple[str, ...] = ("log_level",),
) -> list[dict]:
    """
    Создаёт спецификации CatBoost из исходного пайплайна под выбранный тип устройства.

    Parameters
    ----------
    task_type : str
        Тип вычислений CatBoost: "GPU" для Colab/CUDA или "CPU" для локальной отладки.
    devices : str, default="0"
        Идентификатор GPU-устройства CatBoost.
    verbose : bool, default=True
        Управляет подробностью вывода CatBoost в fit-параметрах.
    random_seed : int, default=RANDOM_STATE
        Seed для воспроизводимости обучения.
    target_modes : tuple[str, ...], default=("log_level",)
        Целевые преобразования, которые нужно проверить в holdout-пайплайне.

    Returns
    -------
    list[dict]
        Список спецификаций модели в формате run_feb_holdout_selected_split_pipeline_v2.

    Notes
    -----
    По умолчанию запускается только "log_level", потому что это лучший target mode
    из исходного model-catboost.ipynb и именно эта модель нужна для 47 признаков.
    """
    task_type = task_type.upper()
    specs = make_default_candidate_specs_gpu(
        verbose=verbose,
        devices=devices,
        random_seed=random_seed,
    )

    selected_specs = []
    for spec in specs:
        if spec["target_mode"] not in target_modes:
            continue
        spec = copy.deepcopy(spec)
        spec["name"] = spec["name"].replace("_gpu", f"_{task_type.lower()}")
        spec["model_params"]["task_type"] = task_type
        spec["model_params"]["random_seed"] = random_seed
        if task_type == "GPU":
            spec["model_params"]["devices"] = devices
        else:
            spec["model_params"].pop("devices", None)
        selected_specs.append(spec)

    if not selected_specs:
        raise ValueError(f"Нет спецификаций для target_modes={target_modes}.")
    return selected_specs

## 47 признаков и 8 категориальных признаков

In [27]:
# ============================================================
# 1. Фичи, которые были "почти хорошими" по ranked-add отбору
# ============================================================

features_to_try_soft = [
    "log_lag1_x_region_feb_mar_growth_2024",
    "region_feb_mar_growth_2024",
    "lag1_x_global_feb_mar_growth_2024",
    "settlement_feb_mar_growth_2024",
    "lag1_x_trading_area_feb_mar_growth_2024",
    "daily_pto_lag_3",

    # REMOVED: weak SHAP top-5
    # "people_per_household",

    "settlement_poi_score_mean",
    "recent_daily_weighted_123",
    "recent_daily_mean_3_prediction",
]


# ============================================================
# 2. Надёжные фичи из твоего adaptive ranked-add отбора
# ============================================================

reliable_features = [
    "log_daily_pto_lag_1",
    "day_scaled_lag1_prediction",
    "log_recent_daily_median_3_prediction",
    "log_daily_same_transition_last_year_prediction",
    "region",
    "number_of_cash_registers",
    "log_day_scaled_x_region_share_change",
    "day_scaled_x_region_share_change",
    "recent_daily_mean_3",
    "settlement_transition_prediction",
    "log_recent_daily_mean_2_prediction",
    "daily_same_transition_last_year_prediction",
    "recent_mean_anchor_to_day_scaled_ratio",
    "avg_items_per_receipt_lag_3",
    "number_of_households",
    "store_share_region_lag2",
    "blend_day_scaled_and_region_march_growth",

    # REMOVED: weak SHAP top-5
    "region_store_count", ###################
    # "avg_cancellations_lag_3",
    # "has_marketplace_nearby",

    # ПРОБУЮ УБРАТЬ!
    # "avg_items_per_receipt_roll_median_3",
    "log_daily_pto_lag_2",

    # REMOVED: weak SHAP top-5
    "pto_lag_1_to_settlement__trading_area_category_pto_mean_lag_1",

    "log_pto_lag_2",
]


# ============================================================
# 3. Сильные недостающие фичи из модели 90.19
#    Важно: часть из них должна идти в CatBoost как category
# ============================================================

top_missing_features_90_19 = [
    # календарно-категориальный блок
    "prev_month",
    "month_name",
    "month",
    "holiday_month_type",

    # store / geo categorical block
    "new_id",
    "population_city_type",
    "opening_date_category",

    # лаги и динамика
    "log_pto_lag_1",
    "log_pto_lag_3",
    "log_pto_diff_lag_1_2",
    "log_pto_diff_lag_1_3",
    "log_pto_diff_lag_3_6",
    "log_pto_diff_lag_1_12",

    # относительные позиции магазина
    "pto_lag_1_to_region_pto_mean_lag_1",

    # календарная эвристика
    "calendar_growth_multiplier",
    "log_calendar_heuristic_prediction",
    "log_pto_lag_1_to_calendar_heuristic",
]


# ============================================================
# 4. Итоговый список фичей без дублей
#    Важно: drop_candidates_strong сюда НЕ добавляем,
#    потому что эти фичи как раз убрали по weak SHAP top-5.
# ============================================================

selected_features = list(dict.fromkeys(
    features_to_try_soft
    + reliable_features
    + top_missing_features_90_19
))


# ============================================================
# 5. Явные категориальные фичи
#    Именно эти фичи нужно передать в CatBoost как category,
#    а не как обычные числа.
# ============================================================

explicit_cat_features = [
    "new_id",
    "month",
    "prev_month",
    "month_name",
    "population_city_type",
    "region",
    "holiday_month_type",
    "opening_date_category",
]

# Оставляем только те categorical-фичи, которые реально есть в selected_features
explicit_cat_features = [
    f for f in explicit_cat_features
    if f in selected_features
]


# ============================================================
# 6. Sanity check перед запуском
# ============================================================

print("Total selected features:", len(selected_features))
print("Explicit categorical features:", len(explicit_cat_features))
print("Explicit categorical features list:")
print(explicit_cat_features)

print("\nSelected features:")
for i, f in enumerate(selected_features, start=1):
    print(f"{i:03d}. {f}")

assert len(selected_features) == 47
assert len(explicit_cat_features) == 8

Total selected features: 47
Explicit categorical features: 8
Explicit categorical features list:
['new_id', 'month', 'prev_month', 'month_name', 'population_city_type', 'region', 'holiday_month_type', 'opening_date_category']

Selected features:
001. log_lag1_x_region_feb_mar_growth_2024
002. region_feb_mar_growth_2024
003. lag1_x_global_feb_mar_growth_2024
004. settlement_feb_mar_growth_2024
005. lag1_x_trading_area_feb_mar_growth_2024
006. daily_pto_lag_3
007. settlement_poi_score_mean
008. recent_daily_weighted_123
009. recent_daily_mean_3_prediction
010. log_daily_pto_lag_1
011. day_scaled_lag1_prediction
012. log_recent_daily_median_3_prediction
013. log_daily_same_transition_last_year_prediction
014. region
015. number_of_cash_registers
016. log_day_scaled_x_region_share_change
017. day_scaled_x_region_share_change
018. recent_daily_mean_3
019. settlement_transition_prediction
020. log_recent_daily_mean_2_prediction
021. daily_same_transition_last_year_prediction
022. recent_mean

# Optuna

In [28]:
!pip install -q \
optuna

In [32]:
def run_catboost_feb2025_loglevel_optuna_full_pipeline(
    *,
    df=None,
    selected_features=None,
    explicit_cat_features=None,
    feature_builder_func=None,
    feature_builder_kwargs=None,

    target_col="pto",
    id_col="new_id",
    year_col="year",
    month_col="month",
    time_col="year_month_index",
    min_year=2023,

    valid_year=2025,
    valid_month=2,

    n_trials=200,
    early_stopping_rounds=75,

    random_seed=None,
    task_type=None,
    devices=None,

    study_version="v5",
    output_dir_name=None,
    study_name=None,
    storage_filename=None,
    load_if_exists=True,
    enqueue_reference_trial=True,

    verbose=True,
    verbose_fit=False,
    show_tables=True,
):
    """
    Полный самодостаточный Optuna-пайплайн для CatBoost log-level target.

    Делает:
    - один holdout fold: 2025-02;
    - y_model = log(pto);
    - CatBoost loss_function='MAE';
    - Optuna objective = MAPE на восстановленном уровне РТО;
    - безопасно обрабатывает CATBOOST_DEVICES=None;
    - не требует внешнего .py импорта;
    - не требует заранее определённых make_catboost_weighted_cv_folds / run_catboost_optuna_search.
    """

    # ============================================================
    # 0. Imports
    # ============================================================

    import gc
    import json
    import time
    import copy
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import optuna

    from catboost import CatBoostRegressor, Pool

    ns = globals()

    # ============================================================
    # 1. Resolve notebook objects
    # ============================================================

    if df is None:
        if "df" not in ns:
            raise RuntimeError("Не найден df. Передай df явно или создай df в notebook namespace.")
        df = ns["df"]

    if selected_features is None:
        if "selected_features" not in ns:
            raise RuntimeError("Не найден selected_features. Передай selected_features явно.")
        selected_features = ns["selected_features"]

    if explicit_cat_features is None:
        if "explicit_cat_features" not in ns:
            raise RuntimeError("Не найден explicit_cat_features. Передай explicit_cat_features явно.")
        explicit_cat_features = ns["explicit_cat_features"]

    if feature_builder_func is None:
        if "build_retail_features_with_split" not in ns:
            raise RuntimeError("Не найден build_retail_features_with_split.")
        feature_builder_func = ns["build_retail_features_with_split"]

    if feature_builder_kwargs is None:
        if "make_x5_training_builder_kwargs" in ns:
            feature_builder_kwargs = ns["make_x5_training_builder_kwargs"]()
        else:
            feature_builder_kwargs = {}

    if random_seed is None:
        random_seed = ns.get("RANDOM_STATE", 42)

    if task_type is None:
        task_type = ns.get("CATBOOST_TASK_TYPE", "GPU")

    if devices is None:
        devices = ns.get("CATBOOST_DEVICES", None)

    selected_features = list(dict.fromkeys(list(selected_features)))
    explicit_cat_features = list(dict.fromkeys(list(explicit_cat_features or [])))

    # ============================================================
    # 2. Helpers
    # ============================================================

    def _sanitize_task_type_and_devices(task_type_value, devices_value):
        if task_type_value is None:
            task_type_safe = "CPU"
        else:
            task_type_safe = str(task_type_value).strip()

        if task_type_safe == "":
            task_type_safe = "CPU"

        if task_type_safe.upper() == "GPU":
            if devices_value is None:
                devices_safe = "0"
            else:
                devices_safe = str(devices_value).strip()

            if devices_safe.lower() in ["", "none", "null", "nan"]:
                devices_safe = "0"

            return "GPU", devices_safe

        return task_type_safe, None

    def _safe_mape_percent(y_true, y_pred, eps=1e-9):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        denom = np.maximum(np.abs(y_true), eps)
        return float(np.mean(np.abs(y_pred - y_true) / denom) * 100.0)

    def _safe_wape_percent(y_true, y_pred, eps=1e-9):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return float(np.sum(np.abs(y_pred - y_true)) / (np.sum(np.abs(y_true)) + eps) * 100.0)

    def _safe_mae(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return float(np.mean(np.abs(y_pred - y_true)))

    def _safe_rmse(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

    def _safe_bias_percent(y_true, y_pred, eps=1e-9):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return float(np.sum(y_pred - y_true) / (np.sum(np.abs(y_true)) + eps) * 100.0)

    def _competition_score_from_mape(mape):
        m = min(float(mape), 100.0)
        return float(100.0 * ((100.0 - m) / 100.0) ** 2)

    def _calc_level_metrics(y_true, y_pred):
        mape = _safe_mape_percent(y_true, y_pred)
        return {
            "mape": mape,
            "wape": _safe_wape_percent(y_true, y_pred),
            "mae": _safe_mae(y_true, y_pred),
            "rmse": _safe_rmse(y_true, y_pred),
            "bias_pct": _safe_bias_percent(y_true, y_pred),
            "score": _competition_score_from_mape(mape),
        }

    def _json_safe(obj):
        if isinstance(obj, dict):
            return {str(k): _json_safe(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [_json_safe(v) for v in obj]
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            val = float(obj)
            return None if not np.isfinite(val) else val
        if isinstance(obj, float):
            return None if not np.isfinite(obj) else obj
        if obj is pd.NA:
            return None
        return obj

    def _prepare_cat_col(s, missing_token="__MISSING__"):
        out = s.astype("object")
        out = out.where(pd.notna(out), missing_token)
        return out.astype(str)

    def _is_object_like(s):
        return (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_string_dtype(s)
            or isinstance(s.dtype, pd.CategoricalDtype)
        )

    def _suggest_from_space(trial, name, spec):
        if isinstance(spec, list):
            if len(spec) == 1:
                return spec[0]
            return trial.suggest_categorical(name, spec)

        if isinstance(spec, tuple):
            if len(spec) == 0:
                raise ValueError(f"Пустой search spec для параметра {name}")

            if isinstance(spec[0], str):
                mode = spec[0]

                if mode == "float_log":
                    return trial.suggest_float(name, float(spec[1]), float(spec[2]), log=True)

                if mode == "float":
                    if len(spec) >= 4:
                        return trial.suggest_float(name, float(spec[1]), float(spec[2]), step=float(spec[3]))
                    return trial.suggest_float(name, float(spec[1]), float(spec[2]))

                if mode == "int":
                    step = int(spec[3]) if len(spec) >= 4 else 1
                    return trial.suggest_int(name, int(spec[1]), int(spec[2]), step=step)

                if mode == "fixed":
                    return spec[1]

                if mode == "categorical":
                    values = list(spec[1])
                    if len(values) == 1:
                        return values[0]
                    return trial.suggest_categorical(name, values)

                raise ValueError(f"Неизвестный mode={mode} для {name}")

            low, high = spec[0], spec[1]
            step = spec[2] if len(spec) >= 3 else None

            if isinstance(low, int) and isinstance(high, int):
                return trial.suggest_int(name, int(low), int(high), step=int(step) if step is not None else 1)

            if step is None:
                return trial.suggest_float(name, float(low), float(high))

            return trial.suggest_float(name, float(low), float(high), step=float(step))

        return spec

    # ============================================================
    # 3. Fixed search space / reference
    # ============================================================

    search_space = {
        "loss_function": ["MAE"],
        "eval_metric": ["MAE"],
        "bootstrap_type": ["Bayesian"],

        "iterations": (5000, 9500, 250),
        "learning_rate": ("float_log", 0.011, 0.026),
        "depth": [4, 5, 6],

        "l2_leaf_reg": [1, 2, 3, 5, 8, 12, 16, 20, 30, 40],
        "random_strength": (14.0, 30.0),
        "one_hot_max_size": (24, 42),

        "max_ctr_complexity": [1],
        "border_count": [192, 254],
        "has_time": [True],
        "leaf_estimation_iterations": (1, 5),
        "bagging_temperature": (0.0, 0.9),
    }

    stage3_reference_params = {
        "loss_function": "MAE",
        "bootstrap_type": "Bayesian",
        "iterations": 6500,
        "learning_rate": 0.017054252342948984,
        "depth": 5,
        "l2_leaf_reg": 3,
        "random_strength": 21.38148662552484,
        "one_hot_max_size": 29,
        "max_ctr_complexity": 1,
        "border_count": 192,
        "has_time": True,
        "leaf_estimation_iterations": 1,
        "eval_metric": "MAE",
        "bagging_temperature": 0.15112160137554814,
    }

    # ============================================================
    # 4. Safe task/device
    # ============================================================

    task_type_safe, devices_safe = _sanitize_task_type_and_devices(task_type, devices)

    if verbose:
        print("OK: resolved input objects")
        print("n selected_features:", len(selected_features))
        print("n explicit_cat_features:", len(explicit_cat_features))
        print("task_type raw:", task_type)
        print("devices raw:", devices)
        print("task_type_safe:", task_type_safe)
        print("devices_safe:", devices_safe)

    # ============================================================
    # 5. Build February holdout features
    # ============================================================

    if verbose:
        print("\n" + "=" * 100)
        print(f"Build February holdout: valid={valid_year}-{valid_month:02d}")
        print("=" * 100)

    output = feature_builder_func(
        df,
        split_mode="train_val",
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        val_year=int(valid_year),
        val_month=int(valid_month),
        **dict(feature_builder_kwargs or {}),
    )

    if "train_features" not in output or "val_features" not in output:
        raise KeyError("feature_builder_func должен вернуть train_features и val_features.")

    train_frame = output["train_features"].copy()
    valid_frame = output["val_features"].copy()

    if verbose:
        print("train_frame shape:", train_frame.shape)
        print("valid_frame shape:", valid_frame.shape)

    # ============================================================
    # 6. Feature check
    # ============================================================

    missing_train = [f for f in selected_features if f not in train_frame.columns]
    missing_valid = [f for f in selected_features if f not in valid_frame.columns]

    if missing_train or missing_valid:
        raise ValueError(
            "Часть selected_features отсутствует после feature_builder.\n"
            f"missing_train={missing_train}\n"
            f"missing_valid={missing_valid}"
        )

    features = list(selected_features)

    cat_features = []
    cat_features.extend(explicit_cat_features)

    for col in features:
        if col in train_frame.columns and _is_object_like(train_frame[col]):
            cat_features.append(col)

    cat_features = list(dict.fromkeys([c for c in cat_features if c in features]))

    for col in cat_features:
        train_frame[col] = _prepare_cat_col(train_frame[col])
        valid_frame[col] = _prepare_cat_col(valid_frame[col])

    if verbose:
        print("features:", len(features))
        print("cat_features:", len(cat_features))
        print("cat_features list:", cat_features)

    # ============================================================
    # 7. Target transform masks
    # ============================================================

    train_mask = train_frame[target_col].notna() & (train_frame[target_col].astype(float) > 0)
    valid_mask = valid_frame[target_col].notna() & (valid_frame[target_col].astype(float) > 0)

    train_used = train_frame.loc[train_mask].copy()
    valid_used = valid_frame.loc[valid_mask].copy()

    if len(train_used) == 0:
        raise ValueError("Пустой train_used после target mask.")
    if len(valid_used) == 0:
        raise ValueError("Пустой valid_used после target mask.")

    y_train = np.log(train_used[target_col].astype(float).to_numpy())
    y_valid = np.log(valid_used[target_col].astype(float).to_numpy())

    y_valid_level = valid_used[target_col].astype(float).to_numpy()

    train_pool_base = Pool(
        train_used[features],
        y_train,
        cat_features=cat_features,
    )

    valid_pool_base = Pool(
        valid_used[features],
        y_valid,
        cat_features=cat_features,
    )

    if verbose:
        print("n_train:", len(train_used))
        print("n_valid:", len(valid_used))

    # ============================================================
    # 8. Paths / study
    # ============================================================

    if output_dir_name is None:
        output_dir_name = f"outputs_catboost_optuna_feb_only_{study_version}"

    if study_name is None:
        study_name = f"catboost_loglevel_feb2025_holdout_mae_bayesian_{study_version}"

    if storage_filename is None:
        storage_filename = f"catboost_loglevel_feb2025_holdout_{study_version}.sqlite3"

    output_dir = Path(output_dir_name)
    output_dir.mkdir(parents=True, exist_ok=True)

    storage_path = output_dir / storage_filename
    storage_url = f"sqlite:///{storage_path}"

    if verbose:
        print("\nOptuna output_dir:", output_dir)
        print("Optuna study_name:", study_name)
        print("Optuna storage_path:", storage_path)

    sampler = optuna.samplers.TPESampler(
        seed=int(random_seed),
        n_startup_trials=20,
        multivariate=True,
        group=True,
    )

    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        sampler=sampler,
        pruner=optuna.pruners.NopPruner(),
        storage=storage_url,
        load_if_exists=bool(load_if_exists),
    )

    if enqueue_reference_trial:
        try:
            study.enqueue_trial(stage3_reference_params, skip_if_exists=True)
        except TypeError:
            if len(study.trials) == 0:
                study.enqueue_trial(stage3_reference_params)

    # ============================================================
    # 9. Objective
    # ============================================================

    def suggest_params(trial):
        params = {
            "loss_function": _suggest_from_space(trial, "loss_function", search_space["loss_function"]),
            "eval_metric": _suggest_from_space(trial, "eval_metric", search_space["eval_metric"]),
            "bootstrap_type": _suggest_from_space(trial, "bootstrap_type", search_space["bootstrap_type"]),

            "iterations": _suggest_from_space(trial, "iterations", search_space["iterations"]),
            "learning_rate": _suggest_from_space(trial, "learning_rate", search_space["learning_rate"]),
            "depth": _suggest_from_space(trial, "depth", search_space["depth"]),

            "l2_leaf_reg": _suggest_from_space(trial, "l2_leaf_reg", search_space["l2_leaf_reg"]),
            "random_strength": _suggest_from_space(trial, "random_strength", search_space["random_strength"]),
            "one_hot_max_size": _suggest_from_space(trial, "one_hot_max_size", search_space["one_hot_max_size"]),

            "max_ctr_complexity": _suggest_from_space(trial, "max_ctr_complexity", search_space["max_ctr_complexity"]),
            "border_count": _suggest_from_space(trial, "border_count", search_space["border_count"]),
            "has_time": _suggest_from_space(trial, "has_time", search_space["has_time"]),
            "leaf_estimation_iterations": _suggest_from_space(
                trial,
                "leaf_estimation_iterations",
                search_space["leaf_estimation_iterations"],
            ),
            "bagging_temperature": _suggest_from_space(
                trial,
                "bagging_temperature",
                search_space["bagging_temperature"],
            ),

            "grow_policy": "SymmetricTree",
            "random_seed": int(random_seed),
            "allow_writing_files": False,
            "task_type": task_type_safe,
            "metric_period": 100,
            "verbose": False,
        }

        params["loss_function"] = "MAE"
        params["eval_metric"] = "MAE"
        params["bootstrap_type"] = "Bayesian"
        params.pop("subsample", None)

        if task_type_safe.upper() == "GPU":
            params["devices"] = devices_safe if devices_safe is not None else "0"
        else:
            params.pop("devices", None)

        if str(params.get("devices", "")).strip().lower() in ["none", "null", "nan"]:
            params["devices"] = "0"

        trial.set_user_attr("target_mode", "log_level")
        trial.set_user_attr("task_type", params.get("task_type"))
        trial.set_user_attr("devices", params.get("devices", None))
        trial.set_user_attr("catboost_loss_function", "MAE")
        trial.set_user_attr("catboost_eval_metric", "MAE")
        trial.set_user_attr("bootstrap_type", "Bayesian")

        return params

    def objective(trial):
        start_time = time.time()

        params = suggest_params(trial)

        fit_params = {
            "early_stopping_rounds": int(early_stopping_rounds),
            "use_best_model": True,
            "verbose": verbose_fit,
        }

        model = None

        try:
            model = CatBoostRegressor(**params)
            model.fit(
                train_pool_base,
                eval_set=valid_pool_base,
                **fit_params,
            )

            pred_log = model.predict(valid_pool_base)
            pred_level = np.exp(np.asarray(pred_log, dtype=float))
            pred_level = np.where(np.isfinite(pred_level), pred_level, np.nan)
            pred_level = np.maximum(pred_level, 0.0)

            metrics = _calc_level_metrics(y_valid_level, pred_level)

            best_iteration = model.get_best_iteration()
            if best_iteration is None:
                best_iteration = -1

            try:
                best_score = model.get_best_score()
            except Exception:
                best_score = None

            elapsed = float(time.time() - start_time)

            fold_record = {
                "trial_number": int(trial.number),
                "fold_idx": 0,
                "fold_name": "february_2025",
                "valid_year": int(valid_year),
                "valid_month": int(valid_month),
                "weight": 1.0,
                "n_train": int(len(train_used)),
                "n_valid": int(len(valid_used)),
                "n_features": int(len(features)),
                "n_cat_features": int(len(cat_features)),
                "best_iteration": int(best_iteration),
                "best_score_json": json.dumps(_json_safe(best_score), ensure_ascii=False),
                **metrics,
            }

            trial.set_user_attr("weighted_mape", metrics["mape"])
            trial.set_user_attr("weighted_wape", metrics["wape"])
            trial.set_user_attr("weighted_bias_pct", metrics["bias_pct"])
            trial.set_user_attr("weighted_score", metrics["score"])
            trial.set_user_attr("elapsed_seconds", elapsed)
            trial.set_user_attr("n_features", len(features))
            trial.set_user_attr("n_cat_features", len(cat_features))
            trial.set_user_attr("best_iteration", int(best_iteration))

            trial.set_user_attr(
                "features_json",
                json.dumps(_json_safe(features), ensure_ascii=False),
            )
            trial.set_user_attr(
                "cat_features_json",
                json.dumps(_json_safe(cat_features), ensure_ascii=False),
            )
            trial.set_user_attr(
                "model_params_json",
                json.dumps(_json_safe(params), ensure_ascii=False),
            )
            trial.set_user_attr(
                "fit_params_json",
                json.dumps(_json_safe(fit_params), ensure_ascii=False),
            )
            trial.set_user_attr(
                "fold_metrics_json",
                json.dumps(_json_safe([fold_record]), ensure_ascii=False),
            )

            return metrics["mape"]

        except Exception as e:
            trial.set_user_attr("error", repr(e))
            raise optuna.TrialPruned(f"CatBoost trial failed: {repr(e)}")

        finally:
            if model is not None:
                del model
            gc.collect()

    # ============================================================
    # 10. Optimize
    # ============================================================

    study.optimize(
        objective,
        n_trials=int(n_trials),
        timeout=None,
        gc_after_trial=True,
        show_progress_bar=True,
    )

    # ============================================================
    # 11. Collect artifacts
    # ============================================================

    trial_summary_df = study.trials_dataframe()

    fold_metric_records = []
    for t in study.trials:
        raw = t.user_attrs.get("fold_metrics_json")
        if raw is None:
            continue
        try:
            fold_metric_records.extend(json.loads(raw))
        except Exception:
            pass

    fold_metrics_df = pd.DataFrame(fold_metric_records)

    complete_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    if not complete_trials:
        trial_summary_df.to_csv(output_dir / f"{study_name}_trial_summary.csv", index=False)
        fold_metrics_df.to_csv(output_dir / f"{study_name}_fold_metrics.csv", index=False)
        raise RuntimeError(
            "Нет COMPLETE trials. Все trials упали или были pruned. "
            "Посмотри колонку user_attrs_error в trial_summary_df."
        )

    best_trial = min(complete_trials, key=lambda t: t.value)

    trial_summary_df.to_csv(output_dir / f"{study_name}_trial_summary.csv", index=False)
    fold_metrics_df.to_csv(output_dir / f"{study_name}_fold_metrics.csv", index=False)

    best_payload = {
        "best_trial_number": int(best_trial.number),
        "best_february_mape": float(best_trial.value),
        "best_february_score": _competition_score_from_mape(float(best_trial.value)),
        "best_sampled_params": _json_safe(best_trial.params),
        "best_full_catboost_params": json.loads(best_trial.user_attrs["model_params_json"]),
        "best_fit_params": json.loads(best_trial.user_attrs["fit_params_json"]),
        "features_for_model": features,
        "cat_features_for_model": cat_features,
        "valid_year": int(valid_year),
        "valid_month": int(valid_month),
        "task_type_safe": task_type_safe,
        "devices_safe": devices_safe,
    }

    with open(output_dir / f"{study_name}_best.json", "w", encoding="utf-8") as f:
        json.dump(best_payload, f, ensure_ascii=False, indent=2)

    # ============================================================
    # 12. Display top trials
    # ============================================================

    cols_to_show = [
        "number",
        "value",
        "state",

        "params_iterations",
        "params_learning_rate",
        "params_depth",
        "params_l2_leaf_reg",
        "params_random_strength",
        "params_one_hot_max_size",
        "params_border_count",
        "params_leaf_estimation_iterations",
        "params_bagging_temperature",

        "user_attrs_weighted_mape",
        "user_attrs_weighted_wape",
        "user_attrs_weighted_bias_pct",
        "user_attrs_weighted_score",
        "user_attrs_elapsed_seconds",
        "user_attrs_best_iteration",
        "user_attrs_task_type",
        "user_attrs_devices",
    ]

    complete_summary_df = trial_summary_df.query("state == 'COMPLETE'").copy()

    top_trials_df = (
        complete_summary_df
        .sort_values("value")
        [[c for c in cols_to_show if c in complete_summary_df.columns]]
        .head(30)
    )

    if show_tables:
        print("\nTop COMPLETE trials:")
        try:
            display(top_trials_df)
        except NameError:
            print(top_trials_df.to_string(index=False))

        print("\nFold metrics preview:")
        try:
            display(fold_metrics_df.sort_values(["trial_number", "fold_idx"]).head(60))
        except NameError:
            print(fold_metrics_df.sort_values(["trial_number", "fold_idx"]).head(60).to_string(index=False))

    if verbose:
        print("\nBest trial number:", best_trial.number)
        print("Best February MAPE:", best_trial.value)
        print("Best February score:", _competition_score_from_mape(float(best_trial.value)))

        print("\nBest sampled params:")
        print(best_trial.params)

        print("\nBest full CatBoost params:")
        print(json.dumps(best_payload["best_full_catboost_params"], ensure_ascii=False, indent=2))

        print("\nArtifacts saved to:", output_dir.resolve())

    # ============================================================
    # 13. Return
    # ============================================================

    return {
        "study": study,
        "trial_summary_df": trial_summary_df,
        "fold_metrics_df": fold_metrics_df,
        "top_trials_df": top_trials_df,

        "best_trial": best_trial,
        "best_value": best_trial.value,
        "best_payload": best_payload,
        "best_full_catboost_params": best_payload["best_full_catboost_params"],
        "best_fit_params": best_payload["best_fit_params"],

        "features_for_model": features,
        "cat_features_for_model": cat_features,

        "train_frame": train_frame,
        "valid_frame": valid_frame,
        "train_used": train_used,
        "valid_used": valid_used,

        "task_type_safe": task_type_safe,
        "devices_safe": devices_safe,

        "output_dir": output_dir,
        "study_name": study_name,
        "storage_path": storage_path,
    }

In [33]:
catboost_feb_optuna_result = run_catboost_feb2025_loglevel_optuna_full_pipeline(
    n_trials=200,
    early_stopping_rounds=75,
    study_version="v5",
    load_if_exists=True,
)

OK: resolved input objects
n selected_features: 47
n explicit_cat_features: 8
task_type raw: GPU
devices raw: None
task_type_safe: GPU
devices_safe: 0

Build February holdout: valid=2025-02
split_mode: train_val
min_year: 2023
max_known_time_idx: 25
val_time_idx: 25
train shape: (466425, 25)
val shape: (18657, 25)
Generated train_features: (466425, 210)
Generated val_features: (18657, 210)


/tmp/ipykernel_240/2087881552.py:1365: DeprecationWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead



Added integrated anchor features to train/val: 200


/tmp/ipykernel_240/2087881552.py:2665: DeprecationWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead



train_frame shape: (466425, 382)
valid_frame shape: (18657, 382)
features: 47
cat_features: 8
cat_features list: ['new_id', 'month', 'prev_month', 'month_name', 'population_city_type', 'region', 'holiday_month_type', 'opening_date_category']
n_train: 466425
n_valid: 18657

Optuna output_dir: outputs_catboost_optuna_feb_only_v5
Optuna study_name: catboost_loglevel_feb2025_holdout_mae_bayesian_v5
Optuna storage_path: outputs_catboost_optuna_feb_only_v5/catboost_loglevel_feb2025_holdout_v5.sqlite3


...

[I 2026-05-30 15:27:54,447] Trial 0 finished with value: 4.0106168870791565 and parameters: {'iterations': 6500, 'learning_rate': 0.017054252342948984, 'depth': 5, 'l2_leaf_reg': 3, 'random_strength': 21.38148662552484, 'one_hot_max_size': 29, 'border_count': 192, 'leaf_estimation_iterations': 1, 'bagging_temperature': 0.15112160137554814}. Best is trial 0 with value: 4.0106168870791565.
[I 2026-05-30 15:28:04,493] Trial 1 finished with value: 4.400644406495107 and parameters: {'iterations': 6750, 'learning_rate': 0.024920753296101838, 'depth': 4, 'l2_leaf_reg': 16, 'random_strength': 16.93447215765494, 'one_hot_max_size': 29, 'border_count': 192, 'leaf_estimation_iterations': 2, 'bagging_temperature': 0.5506676052501416}. Best is trial 0 with value: 4.0106168870791565.
[I 2026-05-30 15:28:14,971] Trial 2 finished with value: 4.108319971898885 and parameters: {'iterations': 5500, 'learning_rate': 0.014142698291421786, 'depth': 6, 'l2_leaf_reg': 30, 'random_strength': 18.87382030677393,

,number,value,state,params_iterations,params_learning_rate,params_depth,params_l2_leaf_reg,params_random_strength,params_one_hot_max_size,params_border_count,params_leaf_estimation_iterations,params_bagging_temperature,user_attrs_weighted_mape,user_attrs_weighted_wape,user_attrs_weighted_bias_pct,user_attrs_weighted_score,user_attrs_elapsed_seconds,user_attrs_best_iteration,user_attrs_task_type,user_attrs_devices
87,87,4.010,COMPLETE,6750,0.021,5,30,27.370,42,192,1,0.193,4.010,3.555,0.091,92.141,13.040,2167,GPU,0
0,0,4.011,COMPLETE,6500,0.017,5,3,21.381,29,192,1,0.151,4.011,3.550,0.210,92.140,37.304,1745,GPU,0
181,181,4.014,COMPLETE,7750,0.017,5,3,17.077,35,192,1,0.214,4.014,3.593,-0.075,92.132,14.968,2534,GPU,0
125,125,4.015,COMPLETE,8000,0.024,5,12,27.518,39,192,1,0.293,4.015,3.556,0.095,92.131,11.033,1792,GPU,0
188,188,4.026,COMPLETE,7750,0.019,5,3,21.050,36,192,1,0.596,4.026,3.600,0.034,92.110,13.434,2205,GPU,0
168,168,4.027,COMPLETE,7250,0.015,5,3,20.050,27,254,1,0.048,4.027,3.567,0.253,92.109,14.290,2367,GPU,0
89,89,4.032,COMPLETE,7000,0.022,5,30,29.219,39,254,1,0.176,4.032,3.635,-0.324,92.099,14.194,2345,GPU,0
23,23,4.032,COMPLETE,5500,0.018,5,12,22.639,32,192,1,0.136,4.032,3.593,-0.106,92.098,13.388,2255,GPU,0
112,112,4.034,COMPLETE,5750,0.019,5,30,28.373,42,192,1,0.165,4.034,3.627,-0.232,92.096,11.023,1787,GPU,0
64,64,4.034,COMPLETE,9000,0.022,5,30,17.644,24,254,1,0.059,4.034,3.575,0.240,92.095,9.744,1541,GPU,0



Fold metrics preview:


,trial_number,fold_idx,fold_name,valid_year,valid_month,weight,n_train,n_valid,n_features,n_cat_features,best_iteration,best_score_json,mape,wape,mae,rmse,bias_pct,score
0,0,0,february_2025,2025,2,1.000,466425,18657,47,8,1745,"{""learn"": {""MAE"": 0.0444683002693359}, ""valida...",4.011,3.550,3255506.678,6946777.345,0.210,92.140
1,1,0,february_2025,2025,2,1.000,466425,18657,47,8,1559,"{""learn"": {""MAE"": 0.04903655045693305}, ""valid...",4.401,3.891,3568618.997,7399720.091,0.922,91.392
2,2,0,february_2025,2025,2,1.000,466425,18657,47,8,1321,"{""learn"": {""MAE"": 0.045345802681299244}, ""vali...",4.108,3.719,3410544.563,7116443.433,-0.594,91.952
3,3,0,february_2025,2025,2,1.000,466425,18657,47,8,3978,"{""learn"": {""MAE"": 0.04543614662057137}, ""valid...",4.162,3.738,3427806.518,7188097.138,-0.291,91.849
4,4,0,february_2025,2025,2,1.000,466425,18657,47,8,6067,"{""learn"": {""MAE"": 0.044748247139143486}, ""vali...",4.148,3.711,3403358.195,7072612.547,-0.444,91.876
5,5,0,february_2025,2025,2,1.000,466425,18657,47,8,4745,"{""learn"": {""MAE"": 0.045354847543817335}, ""vali...",4.167,3.716,3407896.708,7135219.839,-0.241,91.841
6,6,0,february_2025,2025,2,1.000,466425,18657,47,8,4260,"{""learn"": {""MAE"": 0.04717048644610602}, ""valid...",4.186,3.714,3406686.984,7245690.492,0.372,91.802
7,7,0,february_2025,2025,2,1.000,466425,18657,47,8,970,"{""learn"": {""MAE"": 0.0457195565002412}, ""valida...",4.187,3.786,3472369.992,7238704.762,-0.204,91.801
8,8,0,february_2025,2025,2,1.000,466425,18657,47,8,3011,"{""learn"": {""MAE"": 0.04780613928418288}, ""valid...",4.275,3.874,3553147.407,7330557.357,-0.546,91.632
9,9,0,february_2025,2025,2,1.000,466425,18657,47,8,3188,"{""learn"": {""MAE"": 0.04731181242295117}, ""valid...",4.181,3.726,3416942.368,7214171.983,0.376,91.813



Best trial number: 87
Best February MAPE: 4.009808691926637
Best February score: 92.14116827360522

Best sampled params:
{'iterations': 6750, 'learning_rate': 0.02128577619581686, 'depth': 5, 'l2_leaf_reg': 30, 'random_strength': 27.36974751448375, 'one_hot_max_size': 42, 'border_count': 192, 'leaf_estimation_iterations': 1, 'bagging_temperature': 0.192892090385178}

Best full CatBoost params:
{
  "loss_function": "MAE",
  "eval_metric": "MAE",
  "bootstrap_type": "Bayesian",
  "iterations": 6750,
  "learning_rate": 0.02128577619581686,
  "depth": 5,
  "l2_leaf_reg": 30,
  "random_strength": 27.36974751448375,
  "one_hot_max_size": 42,
  "max_ctr_complexity": 1,
  "border_count": 192,
  "has_time": true,
  "leaf_estimation_iterations": 1,
  "bagging_temperature": 0.192892090385178,
  "grow_policy": "SymmetricTree",
  "random_seed": 42,
  "allow_writing_files": false,
  "task_type": "GPU",
  "metric_period": 100,
  "verbose": false,
  "devices": "0"
}

Artifacts saved to: /kaggle/worki